<a id="nb04-toc"></a>
# Notebook 04 — Daily LST Temporal Analysis
<!-- nb04 -->

We analyse daily land surface temperature in each city's urban core for the study year, from MODIS and
ERA5-Land, detect its persistent heat events, select one event per city and period, and analyse that
event's three zones with MSG-SEVIRI. We keep every result apart by dataset (MODIS- or ERA5-detected)
and period (day, night).

---

## Table of Contents

| Section | Description |
|---------|-------------|
| [§1 — Setup](#nb04-s1) | Imports, config, output directories |
| [§2 — AOI + GEE Init](#nb04-s2) | Load UC geometries; initialise GEE |
| [§2b — Filter Assertion](#nb04-s2b) | Verify 300 k city filter |
| [§3 — MODIS Extraction](#nb04-s3) | UC-only daily MODIS LST (Aqua MYD11A1) |
| [§4 — ERA5 Extraction](#nb04-s4) | UC-only daily ERA5-Land LST |
| [§5 — Hot Event Detection](#nb04-s5) | Within-year P90, day + night independently |
| [§5a — Summary Statistics](#nb04-s5a) | City + cluster daily LST stats (MODIS & ERA5, day & night) |
| [§5a2 — National Daily Summary](#nb04-s5a2) | National aggregate of daily LST stats |
| [§5b — Hot-Day Bar Chart](#nb04-s5b) | Persistent against isolated hot days, per city and cluster |
| [§6 — Full-Year Correlation](#nb04-s6) | ERA5 vs MODIS (MODIS-valid days only) |
| [§6a — Target Diagram](#nb04-s6a) | Multi-city ERA5 vs MODIS target diagram |
| [§6b — Cluster Correlation](#nb04-s6b) | Pooled ERA5 vs MODIS per cluster + hexbin |
| [§7 — Event Selection](#nb04-s7) | EXCELLENT / GOOD / FAIR per city |
| [§7a — Audit Trail](#nb04-s7a) | Per-city JSON audit export |
| [§7b — Heat Event Tables](#nb04-s7b) | City / cluster / national heat event summary CSVs |
| [§7b2 — Tidy Heat Event Table](#nb04-s7b2) | One row per city × dataset × period |
| [§7c — Heat Event Charts](#nb04-s7c) | Error-bar bar charts (composite + 2×8 grid) |
| [§8 — Daily Charts](#nb04-s8) | Two-panel MODIS/ERA5 time-series (print 1, save rest) |
| [§8a — Cluster Charts](#nb04-s8a) | Cluster composite daily charts + event barplot |
| [§8b — Publication 2×8 Grid](#nb04-s8b) | City ghost + cluster median, no text boxes; annotation table |
| [§8c — National Cluster Heat Violin](#nb04-s8c) | Butterfly violin of MODIS hot days — multi-city clusters |
| [§8d — Cluster Violin Grid 5×2](#nb04-s8d) | Within-cluster hot day distribution (multi-city only) |
| [§8e — Cluster Butterfly Violins](#nb04-s8e) | Day against night hot days, clusters of ≥ 3 cities |
| [§9 — SEVIRI Download](#nb04-s9) | National mosaic; clip per city at CR level |
| [§10 — SEVIRI 3-Zone](#nb04-s10) | UC / BRA / CR extraction + SUHII |
| [§10b — Cluster SUHII](#nb04-s10b) | Cluster-level SUHII aggregation |
| [§10c — Master Composite Table](#nb04-s10c) | All NB04 results — city / cluster / national |
| [§10d — National SUHII](#nb04-s10d) | National composite SUHII |
| [§11 — SEVIRI×ERA5 Correlation](#nb04-s11) | Event-window only; MODIS excluded |
| [§11a — SEVIRI Target Diagram](#nb04-s11a) | SEVIRI against ERA5-Land over the event window |
| [§12 — Summary](#nb04-s12) | Output tree |

<a id="nb04-s1"></a>
## §1 — Setup & Configuration

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
import sys, warnings, json, importlib
from pathlib import Path

# ── project root on sys.path ──────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebook' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
import importlib; importlib.reload(config)

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from src import gee_utils, visualization, daily_analysis_utils
importlib.reload(gee_utils)
importlib.reload(visualization)
importlib.reload(daily_analysis_utils)

from src.daily_analysis_utils import (
    detect_hot_days, find_persistent_events, assign_tier,
    select_seviri_event, build_event_json, run_city_event_pipeline,
    compute_correlation_metrics, build_cluster_daily_series,
    aggregate_cluster_events, load_daily_csv, print_event_summary,
)
from src.visualization import (
    create_nb04_event_plot,
    create_cluster_event_frequency_chart,
    create_seviri_era5_scatter,
    plot_target_diagram,
)

# ── output directories ────────────────────────────────────────────────────────
RESULTS_04  = PROJECT_ROOT / 'Results' / '04_temporal_analysis_daily'
RAW_DIR     = RESULTS_04 / 'raw_extractions'
EVENTS_DIR  = RESULTS_04 / 'events'
CHARTS_DIR  = RESULTS_04 / 'charts'
CORR_DIR    = RESULTS_04 / 'correlation'
SEVIRI_DIR  = RESULTS_04 / 'seviri'
CLUSTER_DIR = RESULTS_04 / 'cluster_composites'

for _d in [RAW_DIR, EVENTS_DIR, CHARTS_DIR, CORR_DIR, SEVIRI_DIR, CLUSTER_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── study parameters ──────────────────────────────────────────────────────────
STUDY_YEAR          = config.DAILY_LST_CONFIG['study_year']
HOT_PERCENTILE      = config.DAILY_LST_CONFIG['hot_percentile']
PERSISTENT_THRESHOLD = config.DAILY_LST_CONFIG['persistent_threshold']
SKIP_EXISTING       = config.DAILY_LST_CONFIG['skip_existing']

print(f"Study year          : {STUDY_YEAR}")
print(f"Hot percentile      : P{HOT_PERCENTILE}")
print(f"Persistent threshold: {PERSISTENT_THRESHOLD} consecutive days")
print(f"Skip existing       : {SKIP_EXISTING}")
print(f"Results directory   : {RESULTS_04}")


<a id="nb04-s2"></a>
## §2 — AOI Loading & GEE Initialisation

We load NB01's zone GeoJSONs and take each city's **urban core**, the unit of every MODIS and ERA5
extraction in this notebook.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── load cluster lookup ───────────────────────────────────────────────────────
CLUSTER_CSV = PROJECT_ROOT / 'Results' / '01_aoi_delineation' / 'city_cluster25_lookup.csv'
df_cluster  = pd.read_csv(CLUSTER_CSV)
_city_to_cluster = dict(zip(df_cluster['city_name'], df_cluster['Cluster25'].astype(int)))
print(f"Cluster lookup: {len(_city_to_cluster)} cities across {len(set(_city_to_cluster.values()))} clusters")

# ── scan per-city AOI GeoJSONs from Results/01_aoi_delineation/{City}/ ────────
# Folder names may use underscores for spaces (e.g. Port_Harcourt).
# File names match folder names: {FolderName}_urban_core_4326_v5.geojson etc.
# Invalid geometries are repaired with .buffer(0) before union (fixes
# TopologyException on Lagos, Port Harcourt, Sokoto, Uyo BRA/CR polygons).

AOI_DELINEATION_DIR = PROJECT_ROOT / 'Results' / '01_aoi_delineation'

_uc_geom  = {}   # canonical city name -> shapely geometry (WGS84)
_bra_geom = {}
_cr_geom  = {}

_ZONE_SUFFIX = {
    'urban_core':  '_urban_core_4326_v5.geojson',
    'bra':         '_bra_4326_v5.geojson',
    'city_region': '_city_region_4326_v5.geojson',
}
_zone_stores = {'urban_core': _uc_geom, 'bra': _bra_geom, 'city_region': _cr_geom}

def _safe_union(gdf):
    """Union all geometries in a GeoDataFrame; repair invalid ones first."""
    geoms = [g.buffer(0) if not g.is_valid else g for g in gdf.geometry if g is not None]
    if not geoms:
        return None
    if len(geoms) == 1:
        return geoms[0]
    from shapely.ops import unary_union
    return unary_union(geoms)

# Build folder_name -> canonical_city_name mapping
# Canonical name = what the cluster lookup uses (spaces, not underscores).
_folder_to_city = {}
for city_name in _city_to_cluster:
    folder_candidate = city_name.replace(' ', '_')
    _folder_to_city[folder_candidate] = city_name   # e.g. Port_Harcourt -> Port Harcourt
    _folder_to_city[city_name] = city_name           # exact match also works

_missing_geojson = []
for city_dir in sorted(AOI_DELINEATION_DIR.iterdir()):
    if not city_dir.is_dir() or city_dir.name.startswith('.'):
        continue
    folder_name = city_dir.name
    city = _folder_to_city.get(folder_name, folder_name)   # canonical name

    for zone, suffix in _ZONE_SUFFIX.items():
        f = city_dir / (folder_name + suffix)
        if not f.exists():
            continue
        try:
            gdf_z = gpd.read_file(str(f))
            geom  = _safe_union(gdf_z)
            if geom is not None:
                _zone_stores[zone][city] = geom
        except Exception as exc:
            print(f"  WARN {city} [{zone}]: {exc}")

    uc_f = city_dir / (folder_name + '_urban_core_4326_v5.geojson')
    if not uc_f.exists() and city not in _uc_geom:
        _missing_geojson.append(city)

CITIES = sorted(_uc_geom.keys())
print(f"Cities with UC geometry : {len(CITIES)}")
print(f"Cities with BRA geometry: {len(_bra_geom)}")
print(f"Cities with CR geometry : {len(_cr_geom)}")
if _missing_geojson:
    print(f"Absorbed cities (deduplicated): {sorted(set(_missing_geojson))}")
print("First 5:", CITIES[:5])


---
<a id="nb04-s2b"></a>
### §2b — City Filter Assertion

We check the cities against the 300,000 study set, using NB01's `study_cities_final.csv` as the
authority when it exists, and otherwise reporting the configured mode.

> ↑ [Back to Table of Contents](#nb04-toc)

In [ ]:
# ── §2b — 300 k filter assertion ─────────────────────────────────────────────
_final_csv = PROJECT_ROOT / 'Results' / '01_aoi_delineation' / 'study_cities_final.csv'
if _final_csv.exists():
    _expected = set(pd.read_csv(_final_csv).query('status=="included"')['city_name'])
    _loaded   = set(CITIES)
    _extra    = _loaded - _expected
    _missing  = _expected - _loaded
    if _extra:
        print(f'  WARN: {len(_extra)} cities in CITIES NOT in study_cities_final: {sorted(_extra)}')
    if _missing:
        print(f'  WARN: {len(_missing)} expected cities NOT in CITIES: {sorted(_missing)}')
    if not _extra and not _missing:
        print(f'  ✓ Filter verified: {len(CITIES)} cities match study_cities_final.csv')
else:
    _mode = getattr(config, 'CITY_SELECTION_MODE', 'ABOVE_300K')
    print(f'  NOTE: study_cities_final.csv not yet saved (NB01 pending).')
    print(f'        Config mode = {_mode}.  CITIES loaded: {len(CITIES)} cities.')
    print(f'        First 10: {sorted(CITIES)[:10]}')
assert len(CITIES) > 0, 'CITIES list is empty — check AOI GeoJSON directory.'
print(f'  Assertion passed: {len(CITIES)} study cities loaded.')

In [ ]:
# ── Initialise GEE ────────────────────────────────────────────────────────────
import ee
gee_utils.initialize_gee()
print("GEE ready.")


<a id="nb04-s3"></a>
## §3 — MODIS UC Daily LST Extraction

We extract daily MODIS Aqua (MYD11A1) day and night LST over the urban core only, with the strict then
moderate QA levels, into one CSV per city. We summarise the MODIS-detected events separately from the
ERA5-detected ones (§5).

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
from tqdm.notebook import tqdm

def _csv_is_valid(path):
    """Non-empty CSV with a date column."""
    try:
        import pandas as _pd
        df = _pd.read_csv(path, nrows=5)
        return 'date' in df.columns and len(df) > 0
    except Exception:
        return False

# ── MODIS extraction — UC geometry only ───────────────────────────────────────
_modis_status = {}
_pbar = tqdm(CITIES, desc='MODIS extraction', unit='city')

for city in _pbar:
    _pbar.set_postfix_str(city[:24])
    geom_shp = _uc_geom.get(city)
    if geom_shp is None:
        _modis_status[city] = 'no_geometry'
        continue

    out_csv = RAW_DIR / f'{city}_MODIS_UC_daily_{STUDY_YEAR}.csv'
    if SKIP_EXISTING and out_csv.exists() and _csv_is_valid(out_csv):
        _modis_status[city] = 'cached'
        continue

    ee_geom = ee.Geometry(geom_shp.__geo_interface__)
    ok = gee_utils.extract_modis_daily_lst(ee_geom, city, STUDY_YEAR, str(out_csv))
    _modis_status[city] = 'ok' if ok else 'error'

_pbar.close()
_ok = sum(v == 'ok'          for v in _modis_status.values())
_ca = sum(v == 'cached'      for v in _modis_status.values())
_er = sum(v == 'error'       for v in _modis_status.values())
_ng = sum(v == 'no_geometry' for v in _modis_status.values())
print(f'MODIS done — extracted: {_ok}  cached: {_ca}  error: {_er}  no-geom: {_ng}')
if _er:
    print('  Errors:', [c for c, v in _modis_status.items() if v == 'error'])


<a id="nb04-s4"></a>
## §4 — ERA5-Land UC Daily LST Extraction

We extract the daily ERA5-Land skin temperature over the urban core (day 13–15 UTC, night 01–03 UTC).
ERA5-Land's cells are about 9 km across here, so a small core may hold only one or two of them; we note
this and keep those cities.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── ERA5-Land extraction — UC geometry only ───────────────────────────────────
_era5_status = {}
_pbar = tqdm(CITIES, desc='ERA5 extraction', unit='city')

for city in _pbar:
    _pbar.set_postfix_str(city[:24])
    geom_shp = _uc_geom.get(city)
    if geom_shp is None:
        _era5_status[city] = 'no_geometry'
        continue

    out_csv = RAW_DIR / f'{city}_ERA5_UC_daily_{STUDY_YEAR}.csv'
    if SKIP_EXISTING and out_csv.exists() and _csv_is_valid(out_csv):
        _era5_status[city] = 'cached'
        continue

    ee_geom = ee.Geometry(geom_shp.__geo_interface__)
    ok = gee_utils.extract_era5_daily_lst(ee_geom, city, STUDY_YEAR, str(out_csv))
    _era5_status[city] = 'ok' if ok else 'error'

_pbar.close()
_ok = sum(v == 'ok'          for v in _era5_status.values())
_ca = sum(v == 'cached'      for v in _era5_status.values())
_er = sum(v == 'error'       for v in _era5_status.values())
_ng = sum(v == 'no_geometry' for v in _era5_status.values())
print(f'ERA5 done — extracted: {_ok}  cached: {_ca}  error: {_er}  no-geom: {_ng}')
if _er:
    print('  Errors:', [c for c, v in _era5_status.items() if v == 'error'])


<a id="nb04-s5"></a>
## §5 — Within-Year P90 Hot Event Detection

For each city and dataset (MODIS, ERA5), day and night separately, we take the within-year 90th
percentile of the valid observations as the threshold, flag every day at or above it as hot, and call
a run of at least `PERSISTENT_THRESHOLD` consecutive hot days a persistent event. We keep the
MODIS-detected and ERA5-detected summaries apart.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── Load extracted CSVs and run hot-event detection ──────────────────────────
print("Hot Event Detection — P{} threshold, persistent ≥ {} days".format(
      HOT_PERCENTILE, PERSISTENT_THRESHOLD))
print("=" * 70)

# Storage: city → period → audit dict (from run_city_event_pipeline)
# NOTE: audit dicts here contain ONLY detection results (no tier yet — §7 adds tiers)
_modis_dfs  = {}   # city -> full modis DataFrame
_era5_dfs   = {}   # city -> full era5 DataFrame

# Separate summary rows for MODIS-detected and ERA5-detected
_modis_detect_rows = []
_era5_detect_rows  = []

for city in CITIES:
    modis_csv = RAW_DIR / f'{city}_MODIS_UC_daily_{STUDY_YEAR}.csv'
    era5_csv  = RAW_DIR / f'{city}_ERA5_UC_daily_{STUDY_YEAR}.csv'

    # Load
    mdf = load_daily_csv(modis_csv, 'day') if modis_csv.exists() else None
    edf = load_daily_csv(era5_csv,  'day') if era5_csv.exists()  else None

    if mdf is not None:
        _modis_dfs[city] = mdf
    if edf is not None:
        _era5_dfs[city] = edf

    # ── MODIS detection (day and night) ───────────────────────────────────────
    for period in ('day', 'night'):
        mdf_p = load_daily_csv(modis_csv, period) if modis_csv.exists() else None
        if mdf_p is not None and 'lst_median' in mdf_p.columns:
            hot_mask, thr = detect_hot_days(mdf_p['lst_median'], HOT_PERCENTILE)
            events = find_persistent_events(hot_mask, mdf_p['date'], PERSISTENT_THRESHOLD)
            n_hot  = int(hot_mask.sum())
            _modis_detect_rows.append({
                'city': city, 'period': period,
                'p90_threshold_C': round(float(thr), 2) if np.isfinite(thr) else None,
                'n_hot_days':      n_hot,
                'n_persistent_events': len(events),
                'total_persistent_days': sum(e['duration'] for e in events),
            })

    # ── ERA5 detection (day and night) ────────────────────────────────────────
    for period in ('day', 'night'):
        edf_p = load_daily_csv(era5_csv, period) if era5_csv.exists() else None
        if edf_p is not None and 'lst_median' in edf_p.columns:
            hot_mask, thr = detect_hot_days(edf_p['lst_median'], HOT_PERCENTILE)
            events = find_persistent_events(hot_mask, edf_p['date'], PERSISTENT_THRESHOLD)
            n_hot  = int(hot_mask.sum())
            _era5_detect_rows.append({
                'city': city, 'period': period,
                'p90_threshold_C': round(float(thr), 2) if np.isfinite(thr) else None,
                'n_hot_days':      n_hot,
                'n_persistent_events': len(events),
                'total_persistent_days': sum(e['duration'] for e in events),
            })

df_modis_detect = pd.DataFrame(_modis_detect_rows)
df_era5_detect  = pd.DataFrame(_era5_detect_rows)

print("\n── MODIS-detected summary (cities × periods with valid data) ─────────")
print(df_modis_detect.groupby('period')[['n_hot_days','n_persistent_events']].describe().round(1).to_string())
print("\n── ERA5-detected summary ───────────────────────────────────────────────")
print(df_era5_detect.groupby('period')[['n_hot_days','n_persistent_events']].describe().round(1).to_string())

# Export
df_modis_detect.to_csv(EVENTS_DIR / 'MODIS_hot_event_detection_summary.csv', index=False)
df_era5_detect.to_csv(EVENTS_DIR  / 'ERA5_hot_event_detection_summary.csv',  index=False)
print("\n  ✓ Detection summary CSVs saved.")


<a id="nb04-s5a"></a>
### §5a — Summary Statistics: Full-Year Daily LST

We summarise the full-year daily LST per city and per cluster, for each dataset (MODIS, ERA5-Land) and
period (day, night). A city row gives the LST range, mean and spread, the P90 threshold, the valid
observations, the hot days, the persistent events, their days, the longest and the mean duration; a
cluster row gives the minimum, maximum and mean of these across its cities.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── §5a — Full-year daily LST summary statistics ──────────────────────────────
# 4 combinations per city: MODIS day | MODIS night | ERA5 day | ERA5 night
# city-level: 4 rows × n_cities;  cluster-level: 4 rows × n_clusters

_DATASETS = {
    'MODIS':     lambda city: RAW_DIR / f'{city}_MODIS_UC_daily_{STUDY_YEAR}.csv',
    'ERA5-Land': lambda city: RAW_DIR / f'{city}_ERA5_UC_daily_{STUDY_YEAR}.csv',
}
_PERIODS = ('day', 'night')

_city_stat_rows    = []
_cluster_stat_rows = []

for city in CITIES:
    for dataset, csv_fn in _DATASETS.items():
        csv_path = csv_fn(city)
        if not csv_path.exists():
            continue

        for period in _PERIODS:
            df_p = load_daily_csv(csv_path, period)
            if df_p is None or 'lst_median' not in df_p.columns:
                continue

            lst = df_p['lst_median'].dropna()
            if lst.empty:
                continue

            # ── basic LST stats ───────────────────────────────────────────────
            lst_min  = float(lst.min())
            lst_max  = float(lst.max())
            lst_mean = float(lst.mean())
            lst_std  = float(lst.std())
            n_valid  = int(lst.count())

            # ── hot-day detection ─────────────────────────────────────────────
            hot_mask, p90_thr = detect_hot_days(df_p['lst_median'], HOT_PERCENTILE)
            n_hot = int(hot_mask.sum())

            # ── persistent events ─────────────────────────────────────────────
            events = find_persistent_events(hot_mask, df_p['date'], PERSISTENT_THRESHOLD)
            n_evt  = len(events)
            total_persist_days  = sum(e['duration'] for e in events)
            longest_event_days  = max((e['duration'] for e in events), default=0)
            mean_evt_dur        = round(total_persist_days / n_evt, 1) if n_evt > 0 else 0.0

            _city_stat_rows.append({
                'city':                    city,
                'dataset':                 dataset,
                'period':                  period,
                'lst_min_C':               round(lst_min,  2),
                'lst_max_C':               round(lst_max,  2),
                'lst_mean_C':              round(lst_mean, 2),
                'lst_std_C':               round(lst_std,  2),
                'p90_threshold_C':         round(float(p90_thr), 2),
                'n_valid_obs':             n_valid,
                'n_hot_days':              n_hot,
                'n_persistent_events':     n_evt,
                'total_persistent_days':   total_persist_days,
                'longest_event_days':      longest_event_days,
                'mean_event_duration_days': mean_evt_dur,
            })

df_city_stats = pd.DataFrame(_city_stat_rows)

# ── print city-level summary (transposed for readability) ─────────────────────
print("CITY-LEVEL DAILY LST SUMMARY  (MODIS and ERA5-Land, Day and Night)")
print("=" * 90)
_col_order = [
    'city', 'dataset', 'period',
    'lst_min_C', 'lst_max_C', 'lst_mean_C', 'lst_std_C',
    'p90_threshold_C', 'n_valid_obs',
    'n_hot_days', 'n_persistent_events', 'total_persistent_days',
    'longest_event_days', 'mean_event_duration_days',
]
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.width', 200, 'display.float_format', '{:.2f}'.format):
    print(df_city_stats[_col_order].to_string(index=False))

df_city_stats.to_csv(RESULTS_04 / 'City_daily_LST_summary.csv', index=False)
print(f"\n  ✓ City_daily_LST_summary.csv  ({len(df_city_stats)} rows — "
      f"{df_city_stats['city'].nunique()} cities × "
      f"{df_city_stats['dataset'].nunique()} datasets × "
      f"{df_city_stats['period'].nunique()} periods)")

# ── cluster-level aggregation ─────────────────────────────────────────────────
df_city_stats['cluster_num'] = df_city_stats['city'].map(_city_to_cluster)

_METRICS = {
    'lst_min_C':               ['min', 'max', 'mean'],
    'lst_max_C':               ['min', 'max', 'mean'],
    'lst_mean_C':              ['min', 'max', 'mean'],
    'lst_std_C':               ['mean'],
    'p90_threshold_C':         ['min', 'max', 'mean'],
    'n_valid_obs':             ['mean'],
    'n_hot_days':              ['min', 'max', 'mean'],
    'n_persistent_events':     ['min', 'max', 'mean'],
    'total_persistent_days':   ['min', 'max', 'mean'],
    'longest_event_days':      ['min', 'max', 'mean'],
    'mean_event_duration_days': ['min', 'max', 'mean'],
}

_agg_dict = {col: funcs for col, funcs in _METRICS.items()}
_n_cities = df_city_stats.groupby(['cluster_num', 'dataset', 'period'])['city'].nunique()

df_cl_stats = (
    df_city_stats.dropna(subset=['cluster_num'])
    .groupby(['cluster_num', 'dataset', 'period'])
    .agg(_agg_dict)
    .round(2)
    .reset_index()
)
# Flatten multi-level columns
df_cl_stats.columns = [
    '_'.join(c).strip('_') if isinstance(c, tuple) else c
    for c in df_cl_stats.columns
]
df_cl_stats = df_cl_stats.merge(
    _n_cities.reset_index().rename(columns={'city': 'n_cities'}),
    on=['cluster_num', 'dataset', 'period'], how='left'
)

# Reorder: cluster metadata first
_front_cols = ['cluster_num', 'n_cities', 'dataset', 'period']
_rest_cols  = [c for c in df_cl_stats.columns if c not in _front_cols]
df_cl_stats = df_cl_stats[_front_cols + _rest_cols]

print()
print("CLUSTER-LEVEL DAILY LST SUMMARY  (aggregated across cities)")
print("=" * 90)
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.width', 220, 'display.float_format', '{:.2f}'.format):
    print(df_cl_stats.to_string(index=False))

df_cl_stats.to_csv(RESULTS_04 / 'Cluster_daily_LST_summary.csv', index=False)
print(f"\n  ✓ Cluster_daily_LST_summary.csv  ({len(df_cl_stats)} rows — "
      f"{df_cl_stats['cluster_num'].nunique()} clusters × "
      f"{df_cl_stats['dataset'].nunique()} datasets × "
      f"{df_cl_stats['period'].nunique()} periods)")


---
<a id="nb04-s5a2"></a>
### §5a2 — National Daily LST Summary

We add one national row: the mean, maximum and minimum of the city values (`*_mean`, `*_max`, `*_min`).

**Output:** `Results/04_temporal_analysis_daily/national_daily_LST_summary.csv`

> ↑ [Back to Table of Contents](#nb04-toc)

In [ ]:
# ── §5a2 — National daily LST summary ────────────────────────────────────────
_NAT_DAILY_PATH = RESULTS_04 / 'national_daily_LST_summary.csv'
if _NAT_DAILY_PATH.exists():
    df_nat_daily = pd.read_csv(_NAT_DAILY_PATH)
    print(f'§5a2 cache hit — {_NAT_DAILY_PATH.name}  ({len(df_nat_daily)} rows)')
else:
    _num_cols = [
        'lst_min_C', 'lst_max_C', 'lst_mean_C', 'lst_std_C',
        'p90_threshold_C', 'n_valid_obs', 'n_hot_days',
        'n_persistent_events', 'total_persistent_days',
        'longest_event_days', 'mean_event_duration_days',
    ]
    _nat_rows = []
    for (ds, per), grp in df_city_stats.groupby(['dataset', 'period']):
        r = {'scope': 'NATIONAL', 'dataset': ds, 'period': per,
             'n_cities': grp['city'].nunique()}
        for col in _num_cols:
            vals = grp[col].dropna()
            if len(vals):
                r[f'{col}_mean'] = round(float(vals.mean()), 2)
                r[f'{col}_max']  = round(float(vals.max()),  2)
                r[f'{col}_min']  = round(float(vals.min()),  2)
        _nat_rows.append(r)
    df_nat_daily = pd.DataFrame(_nat_rows)
    df_nat_daily.to_csv(_NAT_DAILY_PATH, index=False)
    print(f'  ✓ {_NAT_DAILY_PATH.name}  ({len(df_nat_daily)} rows, {len(df_nat_daily.columns)} cols)')

with pd.option_context('display.max_columns', None, 'display.width', 200):
    print(df_nat_daily.to_string(index=False))

<a id="nb04-s5b"></a>
### §5b — Stacked Hot-Day Bar Chart

For each city and cluster we split the hot days into days in a persistent event (a run of at least two;
dark bar) and isolated hot days (light bar), day (orange-red) beside night (blue).

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
from src.visualization import create_heat_event_stacked_bar
importlib.reload(visualization)

STACKED_DIR = CHARTS_DIR / 'stacked_hot_days'
STACKED_DIR.mkdir(parents=True, exist_ok=True)

# Cluster column name fix (aggregated df uses _mean suffix)
_df_cl_sb = df_cl_stats.copy()
for _col in ('n_hot_days', 'total_persistent_days'):
    _mean_col = f'{_col}_mean'
    if _mean_col in _df_cl_sb.columns and _col not in _df_cl_sb.columns:
        _df_cl_sb[_col] = _df_cl_sb[_mean_col]

# Cluster-level: A4 landscape — ERA5-Land left | MODIS right
fig_sb_cl = create_heat_event_stacked_bar(
    stats_df   = _df_cl_sb,
    level      = 'cluster',
    output_dir = str(STACKED_DIR),
    dpi        = config.FIGURE_DPI,
)
if fig_sb_cl:
    plt.show()
    print(f'  ✓ hot_day_counts_cluster.svg / .pdf')

# City-level: A4 portrait — ERA5-Land left | MODIS right
fig_sb_city = create_heat_event_stacked_bar(
    stats_df   = df_city_stats,
    level      = 'city',
    output_dir = str(STACKED_DIR),
    dpi        = config.FIGURE_DPI,
)
if fig_sb_city:
    plt.show()
    print(f'  ✓ hot_day_counts_city.svg / .pdf')

print(f'  Charts → {STACKED_DIR}')

# [ALTERNATIVE] One figure per dataset (ERA5 / MODIS separately):
# Pass a single-dataset subset to the function and set output_stem manually.


<a id="nb04-s6"></a>
## §6 — Full-Year ERA5 vs MODIS Correlation

We compare ERA5-Land with MODIS urban-core LST on every day MODIS has a valid observation (about
120–250 per city), giving r, r², bias, RMSE, MAE and n per city and period. The SEVIRI comparison of
§11 uses only the selected event window.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
from src.visualization import create_era5_modis_hexbin
importlib.reload(visualization)

# ── Full-year ERA5 vs MODIS correlation (MODIS-valid days only) ────────────────
# For each city × period: inner-merge on date, drop rows where MODIS is NaN,
# compute r / r² / bias / RMSE / MAE from the paired observations.
print("Full-year ERA5 vs MODIS correlation (MODIS-valid days only)")
print("=" * 70)

HEXBIN_CITY_DIR = CORR_DIR / 'hexbin_city'
HEXBIN_CITY_DIR.mkdir(parents=True, exist_ok=True)

_corr_rows  = []
# Accumulate ALL paired obs for the combined all-city hexbin
_all_paired = {'day':   {'era5': [], 'modis': []},
               'night': {'era5': [], 'modis': []}}

_first_shown = False

for city in tqdm(CITIES, desc='City correlation', unit='city'):
    modis_csv = RAW_DIR / f'{city}_MODIS_UC_daily_{STUDY_YEAR}.csv'
    era5_csv  = RAW_DIR / f'{city}_ERA5_UC_daily_{STUDY_YEAR}.csv'
    if not modis_csv.exists() or not era5_csv.exists():
        continue

    _city_paired = {}

    for period in ('day', 'night'):
        mdf_p = load_daily_csv(modis_csv, period)
        edf_p = load_daily_csv(era5_csv,  period)
        if mdf_p is None or edf_p is None:
            continue

        merged = pd.merge(
            mdf_p[['date', 'lst_median']].rename(columns={'lst_median': 'modis'}),
            edf_p[['date', 'lst_median']].rename(columns={'lst_median': 'era5'}),
            on='date', how='inner',
        ).dropna()

        if merged.empty:
            continue

        metrics = compute_correlation_metrics(merged['modis'], merged['era5'])
        metrics.update({'city': city, 'period': period})
        _corr_rows.append(metrics)

        _city_paired[period] = {
            'era5':    merged['era5'].tolist(),
            'modis':   merged['modis'].tolist(),
            'metrics': metrics,
        }
        # Accumulate for combined hexbin
        _all_paired[period]['era5']  += merged['era5'].tolist()
        _all_paired[period]['modis'] += merged['modis'].tolist()

    # Per-city hexbin (print first, save rest silently)
    if _city_paired:
        out_hex = HEXBIN_CITY_DIR / f'{city}_ERA5_vs_MODIS_hexbin.svg'
        show = not _first_shown
        fig = create_era5_modis_hexbin(
            paired_data  = _city_paired,
            title        = f'{city} — ERA5-Land vs MODIS  ({STUDY_YEAR})',
            output_path  = str(out_hex),
            show_inline  = show,
        )
        if show and fig is not None:
            _first_shown = True

df_corr = pd.DataFrame(_corr_rows)
df_corr.to_csv(CORR_DIR / 'ERA5_vs_MODIS_full_year_correlation.csv', index=False)
print(f"  ✓ ERA5_vs_MODIS_full_year_correlation.csv  ({len(df_corr)} rows)")
print(f"  ✓ Per-city hexbin SVGs → {HEXBIN_CITY_DIR}")

# ── Combined all-city hexbin ───────────────────────────────────────────────────
import numpy as np
_combined_paired = {}
for period in ('day', 'night'):
    e = np.array(_all_paired[period]['era5'])
    m = np.array(_all_paired[period]['modis'])
    if len(e) > 1:
        # POOLED over every MODIS-valid daily pair from all cities — these describe
        # the hexbin cloud itself, so n = total pairs (not city count). The §6a target
        # diagram reports the complementary mean-across-cities values; each is labelled.
        metrics_all = compute_correlation_metrics(pd.Series(m), pd.Series(e))
        _combined_paired[period] = {'era5': e, 'modis': m, 'metrics': metrics_all}

fig_all = create_era5_modis_hexbin(
    paired_data = _combined_paired,
    title       = f'All Cities Pooled — ERA5-Land vs MODIS ({STUDY_YEAR}) — annotations: pooled over all daily pairs (n = total pairs)',
    output_path = str(CORR_DIR / 'ERA5_vs_MODIS_all_cities_hexbin.svg'),
    gridsize    = 45,
    show_inline = True,
)
print("  ✓ ERA5_vs_MODIS_all_cities_hexbin.svg")

# ── Summary: POOLED over all daily pairs (matches the combined hexbin) ─────────
print()
print("Combined hexbin = POOLED over all MODIS-valid daily pairs (n = total pairs):")
for period in ('day', 'night'):
    met = _combined_paired.get(period, {}).get('metrics')
    if not met:
        continue
    print(f"  {period.upper():5s}  r={met['r']:.3f}  r2={met['r2']:.3f}  "
          f"bias={met['bias']:+.2f}°C  RMSE={met['rmse']:.2f}°C  n_obs={met['n']:,}")
print("(The §6a target diagram reports the complementary mean-across-cities values.)")


<a id="nb04-s6a"></a>
### §6a — Target Diagram: ERA5-Land vs MODIS Full-Year Validation

We show the agreement of all cities in one Target Diagram (Jolliff et al., 2009):

| Axis / Feature | Metric | Interpretation |
|---|---|---|
| **x-axis** | Unbiased RMSE (uRMSE, °C) | Random-error component |
| **y-axis** | Bias (°C) | Systematic offset: ERA5-Land − MODIS |
| **Distance from origin** | Total RMSE (°C) | Overall model-observation disagreement |
| **Concentric circles** | iso-RMSE contours | Reference performance rings |
The origin is perfect agreement; a point above the x-axis means ERA5-Land is warmer than MODIS on
average, below means cooler. The x-axis is the signed unbiased RMSE, its sign that of the correlation
r (Jolliff et al., 2009, eq. 3), so a positive position means the two co-vary:

```
uRMSE  =  sign(r) × √( RMSE² − Bias² )
```

Triangles are day and circles night, coloured by Cluster25 class, with callouts for the labels: up to
112 points (56 cities × 2 periods). Cities that sit together share ERA5 error characteristics even when
far apart, reflecting their shared landscape type. We compute the metrics per city and period on the
MODIS-valid days:

```python
Bias  = mean(ERA5_LST) − mean(MODIS_LST)          # systematic error, °C
RMSE  = √[ mean( (ERA5_LST_i − MODIS_LST_i)² ) ]  # total error, °C
r     = Pearson correlation(MODIS, ERA5)            # linear agreement
uRMSE = sign(r) × √( RMSE² − Bias² )              # random error, °C
```
#### References

* Jolliff, J. K., Kindle, J. C., Shulman, we., Penta, B., Friedrichs, M. A. M.,
  Helber, R., & Arnone, R. A. (2009). Summary diagrams for coupled
  hydrodynamic-ecosystem model skill assessment. *Journal of Marine Systems*,
  76(1–2), 64–82. <https://doi.org/10.1016/j.jmarsys.2008.05.014>

* Taylor, K. E. (2001). Summarizing model performance in a single diagram.
  *Journal of Geophysical Research: Atmospheres*, 106(D7), 7183–7192.
  <https://doi.org/10.1029/2000JD900719>

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── §6a — Target Diagram (ERA5-Land vs MODIS, all cities, day + night) ────────
# df_corr was built in §6; we reshape it into the list-of-dicts expected by
# plot_target_diagram().

importlib.reload(visualization)
from src.visualization import plot_target_diagram

def _get(city, period, col, default=float('nan')):
    sub = df_corr[(df_corr['city'] == city) & (df_corr['period'] == period)]
    return float(sub[col].iloc[0]) if not sub.empty and col in sub.columns else default

_target_input = []
for city in sorted(df_corr['city'].unique()):
    _target_input.append({
        'city':              city,
        'day_bias':          _get(city, 'day',   'bias'),
        'day_rmse':          _get(city, 'day',   'rmse'),
        'day_correlation':   _get(city, 'day',   'r'),
        'night_bias':        _get(city, 'night', 'bias'),
        'night_rmse':        _get(city, 'night', 'rmse'),
        'night_correlation': _get(city, 'night', 'r'),
    })

# Drop cities where BOTH day and night are NaN (no valid paired data)
_target_input = [
    d for d in _target_input
    if not (pd.isna(d['day_bias']) and pd.isna(d['night_bias']))
]

fig_td, td_summary = plot_target_diagram(
    comparison_results = _target_input,
    dataset1_name      = 'MODIS',
    dataset2_name      = 'ERA5-Land',
    output_dir         = str(CORR_DIR),
    cluster_lookup     = _city_to_cluster,
    show_labels        = True,
)

plt.show()

print()
print(f"TARGET DIAGRAM SUMMARY  (n={td_summary['n_cities']} cities)")
print("=" * 60)
for period in ('day', 'night'):
    s = td_summary[period]
    print(f"  {period.upper():5s}  mean_bias={s['mean_bias']:+.2f}°C  "
          f"mean_RMSE={s['mean_rmse']:.2f}°C  mean_r={s['mean_correlation']:.3f}  "
          f"best={s['best_city']} ({s['best_rmse']:.2f}°C)  "
          f"worst={s['worst_city']} ({s['worst_rmse']:.2f}°C)")


<a id="nb04-s6b"></a>
### §6b — Cluster-Level ERA5 vs MODIS Correlation

We pool all valid daily ERA5–MODIS pairs of a cluster's cities and compute one r, r², bias and RMSE
from them — not an average of city metrics. A single-city cluster equals its city.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
HEXBIN_CL_DIR = CORR_DIR / 'hexbin_cluster'
HEXBIN_CL_DIR.mkdir(parents=True, exist_ok=True)

# ── Pool paired observations per cluster ──────────────────────────────────────
# Group cities by cluster, read their CSVs, inner-merge MODIS+ERA5 on date,
# concatenate into one DataFrame per cluster × period.
from collections import defaultdict

_cl_city_map = defaultdict(list)
for city, cnum in _city_to_cluster.items():
    if city in CITIES:
        _cl_city_map[int(cnum)].append(city)

_cl_corr_rows = []
_first_cl_shown = False

_cluster_nums = sorted(_cl_city_map.keys())
for cnum in tqdm(_cluster_nums, desc='Cluster correlation', unit='cluster'):
    cities_in_cl = _cl_city_map[cnum]
    _cl_paired = {}

    for period in ('day', 'night'):
        _e_all, _m_all = [], []

        for city in cities_in_cl:
            modis_csv = RAW_DIR / f'{city}_MODIS_UC_daily_{STUDY_YEAR}.csv'
            era5_csv  = RAW_DIR / f'{city}_ERA5_UC_daily_{STUDY_YEAR}.csv'
            if not modis_csv.exists() or not era5_csv.exists():
                continue

            mdf_p = load_daily_csv(modis_csv, period)
            edf_p = load_daily_csv(era5_csv,  period)
            if mdf_p is None or edf_p is None:
                continue

            merged = pd.merge(
                mdf_p[['date', 'lst_median']].rename(columns={'lst_median': 'modis'}),
                edf_p[['date', 'lst_median']].rename(columns={'lst_median': 'era5'}),
                on='date', how='inner',
            ).dropna()
            _e_all  += merged['era5'].tolist()
            _m_all  += merged['modis'].tolist()

        if len(_e_all) < 2:
            continue

        import numpy as np
        e_arr = np.array(_e_all)
        m_arr = np.array(_m_all)
        metrics_cl = compute_correlation_metrics(pd.Series(m_arr), pd.Series(e_arr))
        _cl_corr_rows.append({
            'cluster_num': cnum,
            'n_cities':    len(cities_in_cl),
            'period':      period,
            **metrics_cl,
        })
        _cl_paired[period] = {'era5': e_arr, 'modis': m_arr, 'metrics': metrics_cl}

    # Cluster hexbin (print first, save rest)
    if _cl_paired:
        n_c = len(cities_in_cl)
        city_list = ', '.join(cities_in_cl[:3]) + ('...' if n_c > 3 else '')
        out_hex_cl = HEXBIN_CL_DIR / f'Cluster{cnum:02d}_ERA5_vs_MODIS_hexbin.svg'
        show_cl = not _first_cl_shown
        _cl_unit  = 'city' if n_c == 1 else 'cities'
        _cl_title = f'Cluster {cnum:02d} ({n_c} {_cl_unit}) -- ERA5-Land vs MODIS -- {city_list}'
        fig_cl = create_era5_modis_hexbin(
            paired_data = _cl_paired,
            title       = _cl_title,
            output_path = str(out_hex_cl),
            show_inline = show_cl,
        )
        if show_cl and fig_cl is not None:
            _first_cl_shown = True

df_cl_corr = pd.DataFrame(_cl_corr_rows)
df_cl_corr.to_csv(CORR_DIR / 'Cluster_ERA5_vs_MODIS_correlation.csv', index=False)

print()
print("CLUSTER-LEVEL ERA5 vs MODIS CORRELATION (pooled city observations)")
print("=" * 70)
print(f"{'Cluster':>9s} {'n_cities':>9s} {'Period':>7s} "
      f"{'r':>7s} {'r²':>7s} {'Bias':>8s} {'RMSE':>8s} {'n_obs':>8s}")
print("-" * 70)
for _, row in df_cl_corr.iterrows():
    print(f"  C{int(row['cluster_num']):02d}     {int(row['n_cities']):>9d} {row['period']:>7s} "
          f"{row['r']:>7.3f} {row['r2']:>7.3f} {row['bias']:>+8.2f} "
          f"{row['rmse']:>8.2f} {int(row['n']):>8d}")
print()
print(f"  ✓ Cluster_ERA5_vs_MODIS_correlation.csv  ({len(df_cl_corr)} rows)")
print(f"  ✓ Cluster hexbin SVGs → {HEXBIN_CL_DIR}")


<a id="nb04-s7"></a>
## §7 — EXCELLENT / GOOD / FAIR Event Selection

For each city and period we go through the ERA5 persistent events from the longest, and grade each:
**EXCELLENT** when it overlaps a MODIS persistent event, **GOOD** when it overlaps a single MODIS hot
day, **FAIR** otherwise. We select the best grade, then the longest, then the earliest. The selected
event sets the SEVIRI window (§9).

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── Full event pipeline per city ──────────────────────────────────────────────
print("EXCELLENT / GOOD / FAIR event selection")
print("=" * 70)

_audit_day   = {}   # city -> audit dict for 'day'
_audit_night = {}   # city -> audit dict for 'night'

for city in CITIES:
    modis_csv = RAW_DIR / f'{city}_MODIS_UC_daily_{STUDY_YEAR}.csv'
    era5_csv  = RAW_DIR / f'{city}_ERA5_UC_daily_{STUDY_YEAR}.csv'

    for period, store in [('day', _audit_day), ('night', _audit_night)]:
        mdf_p = load_daily_csv(modis_csv, period) if modis_csv.exists() else None
        edf_p = load_daily_csv(era5_csv,  period) if era5_csv.exists()  else None

        audit = run_city_event_pipeline(
            modis_df       = mdf_p,
            era5_df        = edf_p,
            city           = city,
            period         = period,
            percentile     = HOT_PERCENTILE,
            min_consecutive = PERSISTENT_THRESHOLD,
        )
        store[city] = audit

# Print summary (one line per city × period)
print(f"\n{'City':22s} {'Period':7s} {'ERA5 events':>12s} {'Selected tier':>14s} {'Duration':>9s}")
print("-" * 68)
for city in CITIES:
    for period, store in [('day', _audit_day), ('night', _audit_night)]:
        a = store.get(city, {})
        n_e  = len(a.get('era5_persistent_events', []))
        sel  = a.get('selected_event')
        tier = sel.get('tier', '—') if sel else '—'
        dur  = sel.get('duration', 0) if sel else 0
        print(f"  {city:20s} {period:7s} {n_e:>12d} {tier:>14s} {dur:>8}d")

# Tier distribution
print("\nTier distribution:")
for period, store, label in [('day', _audit_day, 'Day'), ('night', _audit_night, 'Night')]:
    from collections import Counter
    tiers = [a.get('selected_event', {}).get('tier') for a in store.values() if a.get('selected_event')]
    cnt   = Counter(t for t in tiers if t)
    print(f"  {label:6s}: EXCELLENT={cnt.get('EXCELLENT',0)}  "
          f"GOOD={cnt.get('GOOD',0)}  FAIR={cnt.get('FAIR',0)}  "
          f"None (no ERA5 event)={sum(1 for a in store.values() if not a.get('selected_event'))}")


<a id="nb04-s7a"></a>
### §7a — JSON Audit Trail Export

We write one JSON file per city with all its ERA5 events, their grades, the selected event and why it
was chosen.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── Export JSON audit trail ───────────────────────────────────────────────────
print("Exporting JSON audit trails...")
for city in CITIES:
    city_audit = {
        'city':       city,
        'study_year': STUDY_YEAR,
        'day':        _audit_day.get(city, {}),
        'night':      _audit_night.get(city, {}),
    }
    out_json = EVENTS_DIR / f"{city.replace(' ', '_')}_event_audit.json"
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(city_audit, f, ensure_ascii=False, indent=2)

print(f"  ✓ {len(CITIES)} audit JSON files -> {EVENTS_DIR}")

# Cluster-level event aggregate
df_cl_events_day   = aggregate_cluster_events(_audit_day,   _city_to_cluster)
df_cl_events_night = aggregate_cluster_events(_audit_night, _city_to_cluster)

df_cl_events_day['period']   = 'day'
df_cl_events_night['period'] = 'night'
df_cl_events = pd.concat([df_cl_events_day, df_cl_events_night], ignore_index=True)
df_cl_events.to_csv(EVENTS_DIR / 'Cluster_event_selection_summary.csv', index=False)
print("  ✓ Cluster_event_selection_summary.csv")
print()
print(df_cl_events.to_string(index=False))


---
<a id="nb04-s7b"></a>
### §7b — Consolidated Heat Event Summary Tables

We merge the daily statistics of §5a with the audit files of §7a into one table per level:

| Level | Output file |
|-------|-------------|
| City | `city_heat_event_summary.csv` |
| Cluster | `cluster_heat_event_summary.csv` |
| National | `national_heat_event_summary.csv` |
Each holds, for day and night, the MODIS and ERA5-Land LST statistics, P90 threshold, hot days and
persistent events; the selected event (start, end, duration, tier); and the correlation (r, r², bias,
RMSE, MAE). The cluster table adds the spread across cities. We recompute only when all three files are
missing.

> ↑ [Back to Table of Contents](#nb04-toc)

In [ ]:
# ── §7b — Consolidated heat event summary tables ─────────────────────────────
_HEAT_CACHES = [
    RESULTS_04 / 'city_heat_event_summary.csv',
    RESULTS_04 / 'cluster_heat_event_summary.csv',
    RESULTS_04 / 'national_heat_event_summary.csv',
]
if all(p.exists() for p in _HEAT_CACHES):
    print('§7b cache hit — loading heat event summary CSVs')
    df_city_heat = pd.read_csv(_HEAT_CACHES[0])
    df_cl_heat   = pd.read_csv(_HEAT_CACHES[1])
    df_nat_heat  = pd.read_csv(_HEAT_CACHES[2])
    print(f'  city: {len(df_city_heat)} rows | cluster: {len(df_cl_heat)} | national: 1')
else:
    # ── Step 1: pivot df_city_stats → one row per city ────────────────────────
    _STAT_COLS = [
        'lst_min_C', 'lst_max_C', 'lst_mean_C', 'lst_std_C',
        'p90_threshold_C', 'n_valid_obs', 'n_hot_days',
        'n_persistent_events', 'total_persistent_days',
        'longest_event_days', 'mean_event_duration_days',
    ]
    _wide_rows = []
    for city in sorted(df_city_stats['city'].unique()):
        row = {'city': city}
        for ds_key, ds_pfx in [('MODIS', 'modis'), ('ERA5-Land', 'era5')]:
            for per in ('day', 'night'):
                sub = df_city_stats[
                    (df_city_stats['city']    == city) &
                    (df_city_stats['dataset'] == ds_key) &
                    (df_city_stats['period']  == per)
                ]
                pfx = f'{ds_pfx}_{per}'
                if sub.empty:
                    for col in _STAT_COLS:
                        row[f'{pfx}_{col}'] = None
                else:
                    r = sub.iloc[0]
                    for col in _STAT_COLS:
                        row[f'{pfx}_{col}'] = r.get(col)
        _wide_rows.append(row)
    df_wide = pd.DataFrame(_wide_rows)

    # ── Step 2: load per-city audit JSONs ─────────────────────────────────────
    _AUDIT_COLS = [
        'era5_p90_C', 'modis_p90_C',
        'sel_start', 'sel_end', 'sel_duration_days', 'sel_tier',
        'n_era5_persist_evts', 'era5_longest_days', 'n_modis_hot_dates',
        'corr_n', 'corr_r', 'corr_r2', 'corr_bias', 'corr_rmse', 'corr_mae',
    ]
    _audit_rows = []
    for city in df_wide['city']:
        fname = city.replace(' ', '_') + '_event_audit.json'
        jpath = EVENTS_DIR / fname
        row = {'city': city}
        for per in ('day', 'night'):
            if jpath.exists():
                jdata = json.load(open(jpath, encoding='utf-8'))
                pdata = jdata.get(per, {})
                thr   = pdata.get('thresholds', {})
                sel   = pdata.get('selected_event') or {}
                corr  = pdata.get('correlation_metrics', {})
                e5e   = pdata.get('era5_persistent_events', [])
                row[f'{per}_era5_p90_C']          = thr.get('era5_p90_celsius')
                row[f'{per}_modis_p90_C']          = thr.get('modis_p90_celsius')
                row[f'{per}_sel_start']            = sel.get('start_date')
                row[f'{per}_sel_end']              = sel.get('end_date')
                row[f'{per}_sel_duration_days']    = sel.get('duration')
                row[f'{per}_sel_tier']             = sel.get('tier')
                _e5_durs  = [e['duration'] for e in e5e]
                _mod_evts = pdata.get('modis_persistent_events', [])
                _mod_durs = [e['duration'] for e in _mod_evts]
                import statistics as _stats
                row[f'{per}_n_era5_persist_evts']   = len(_e5_durs)
                row[f'{per}_era5_longest_days']     = max(_e5_durs, default=0)
                row[f'{per}_era5_shortest_days']    = min(_e5_durs, default=0)
                row[f'{per}_era5_mean_event_days']  = (round(sum(_e5_durs) / len(_e5_durs), 2)
                                                        if _e5_durs else 0.0)
                row[f'{per}_era5_std_event_days']   = (round(_stats.stdev(_e5_durs), 2)
                                                        if len(_e5_durs) > 1 else 0.0)
                row[f'{per}_n_modis_persist_evts']  = len(_mod_durs)
                row[f'{per}_modis_longest_days']    = max(_mod_durs, default=0)
                row[f'{per}_modis_shortest_days']   = min(_mod_durs, default=0)
                row[f'{per}_modis_mean_event_days'] = (round(sum(_mod_durs) / len(_mod_durs), 2)
                                                        if _mod_durs else 0.0)
                row[f'{per}_modis_std_event_days']  = (round(_stats.stdev(_mod_durs), 2)
                                                        if len(_mod_durs) > 1 else 0.0)
                row[f'{per}_n_modis_hot_dates']     = pdata.get('n_modis_hot_dates', 0)
                row[f'{per}_corr_n']               = corr.get('n')
                row[f'{per}_corr_r']               = corr.get('r')
                row[f'{per}_corr_r2']              = corr.get('r2')
                row[f'{per}_corr_bias']            = corr.get('bias')
                row[f'{per}_corr_rmse']            = corr.get('rmse')
                row[f'{per}_corr_mae']             = corr.get('mae')
            else:
                for c in (_AUDIT_COLS + [
                    'era5_shortest_days', 'era5_mean_event_days', 'era5_std_event_days',
                    'n_modis_persist_evts', 'modis_longest_days', 'modis_shortest_days',
                    'modis_mean_event_days', 'modis_std_event_days',
                ]):
                    row[f'{per}_{c}'] = None
        _audit_rows.append(row)
    df_audit = pd.DataFrame(_audit_rows)

    # ── Step 3: merge + cluster + validation status ───────────────────────────
    df_city_heat = df_wide.merge(df_audit, on='city', how='left')
    df_city_heat['cluster25'] = (
        df_city_heat['city'].map(_city_to_cluster).astype('Int64')
    )
    # Derive per-city validation status from city's own day/night audit tiers
    # Rule: EXCELLENT only if both day+night EXCELLENT; FAIR if either FAIR; else GOOD
    _TIER_ORD = {'FAIR': 0, 'GOOD': 1, 'EXCELLENT': 2}
    def _city_vstat(row):
        ts = [t for t in [row.get('day_sel_tier'), row.get('night_sel_tier')]
              if isinstance(t, str) and t in _TIER_ORD]
        if not ts:
            return None
        return min(ts, key=lambda t: _TIER_ORD[t])
    df_city_heat['validation_status'] = df_city_heat.apply(_city_vstat, axis=1)
    _front = ['city', 'cluster25', 'validation_status']
    df_city_heat = df_city_heat[
        _front + [c for c in df_city_heat.columns if c not in _front]
    ]
    df_city_heat.to_csv(_HEAT_CACHES[0], index=False)
    print(f'  ✓ {_HEAT_CACHES[0].name}  '
          f'({len(df_city_heat)} cities, {len(df_city_heat.columns)} cols)')

    # ── Step 4: cluster-level aggregation ─────────────────────────────────────
    _num_city_cols = [
        c for c in df_city_heat.select_dtypes('number').columns
        if c != 'cluster25'
    ]
    _cl_rows = []
    for cnum in sorted(df_city_heat['cluster25'].dropna().unique()):
        sub  = df_city_heat[df_city_heat['cluster25'] == cnum]
        crow = {
            'cluster25':           int(cnum),
            'n_cities':            len(sub),
            'city_names':          ', '.join(sorted(sub['city'])),
            'modal_validation_tier': (
                sub['validation_status'].mode().iloc[0]
                if not sub['validation_status'].isna().all() else None
            ),
        }
        for col in _num_city_cols:
            vals = sub[col].dropna()
            if len(vals):
                crow[f'{col}_mean'] = round(float(vals.mean()), 3)
                crow[f'{col}_max']  = round(float(vals.max()),  3)
                crow[f'{col}_min']  = round(float(vals.min()),  3)
        _cl_rows.append(crow)
    df_cl_heat = pd.DataFrame(_cl_rows)
    df_cl_heat.to_csv(_HEAT_CACHES[1], index=False)
    print(f'  ✓ {_HEAT_CACHES[1].name}  '
          f'({len(df_cl_heat)} clusters, {len(df_cl_heat.columns)} cols)')

    # ── Step 5: national row ──────────────────────────────────────────────────
    _nat = {
        'scope':      'NATIONAL',
        'n_cities':   len(df_city_heat),
        'n_clusters': int(df_city_heat['cluster25'].nunique()),
    }
    for col in _num_city_cols:
        vals = df_city_heat[col].dropna()
        if len(vals):
            _nat[f'{col}_mean'] = round(float(vals.mean()), 3)
            _nat[f'{col}_max']  = round(float(vals.max()),  3)
            _nat[f'{col}_min']  = round(float(vals.min()),  3)
    df_nat_heat = pd.DataFrame([_nat])
    df_nat_heat.to_csv(_HEAT_CACHES[2], index=False)
    print(f'  ✓ {_HEAT_CACHES[2].name}  (1 row, {len(df_nat_heat.columns)} cols)')

print()
_display_cols = ['city', 'cluster25', 'validation_status',
                 'modis_day_n_hot_days', 'modis_night_n_hot_days',
                 'modis_day_lst_mean_C', 'modis_night_lst_mean_C',
                 'day_sel_start', 'day_sel_end', 'day_sel_tier',
                 'night_sel_start', 'night_sel_end', 'night_sel_tier']
_disp = [c for c in _display_cols if c in df_city_heat.columns]
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.width', 240):
    print(df_city_heat[_disp].to_string(index=False))

<a id="nb04-s7b2"></a>
### §7b2 — Tidy Long-Format Heat Event Table

We reshape the city table of §7b into one row per city × dataset × period:

| Identifier | Meaning |
|-----------|---------|
| `cluster25` | Landscape cluster **name/id** (1, 4, 23 …) |
| `cluster_label` | Canonical chart/map label `C{N}` |
| `cluster_profile` | Human-readable Climate / Landcover / Elevation classes |
| `city` | City name |
| `dataset` | `MODIS` or `ERA5-Land` |
| `period` | `day` or `night` |
Each row holds `n_hot_days`, `n_persistent_events`, `total_persistent_days`, `longest_event_days`,
`mean_event_duration_days`, `p90_threshold_C` and `lst_mean_C`. The selected-event columns (`sel_start`,
`sel_end`, `sel_duration_days`, `sel_tier`, `validation_status`) belong to the city and period, so they
repeat on both dataset rows.

**Output:** `heat_event_summary_long.csv`

> ↑ [Back to Table of Contents](#nb04-toc)

In [ ]:
# ── §7b2 — Tidy long-format heat event table (city × dataset × period) ───────
# Melts the wide §7b city_heat_event_summary into one row per city × dataset
# × period, with an explicit `period` column. cluster25 IS the cluster name.
_LONG_OUT = RESULTS_04 / 'heat_event_summary_long.csv'

# Readable physiographic profile per cluster (from the Cluster25 lookup)
_cl_profile = {}
for _, _cr in df_cluster.iterrows():
    _parts = [str(_cr[c]) for c in ('Climate_Class', 'Landcover_Class', 'Elevation_Class')
              if c in df_cluster.columns and pd.notna(_cr[c])]
    _cl_profile[int(_cr['Cluster25'])] = ' / '.join(_parts)

_STATS_LONG = [
    'n_hot_days', 'n_persistent_events', 'total_persistent_days',
    'longest_event_days', 'mean_event_duration_days',
    'p90_threshold_C', 'lst_mean_C',
]

_long_rows = []
for _, r in df_city_heat.iterrows():
    city = r['city']
    cl   = r['cluster25']
    cl_i = int(cl) if pd.notna(cl) else None
    for ds_pfx, ds_name in [('modis', 'MODIS'), ('era5', 'ERA5-Land')]:
        for per in ('day', 'night'):
            row = {
                'cluster25':       cl_i,
                'cluster_label':   (f'C{cl_i}' if cl_i is not None else None),
                'cluster_profile': _cl_profile.get(cl_i),
                'city':            city,
                'dataset':         ds_name,
                'period':          per,
            }
            for s in _STATS_LONG:
                row[s] = r.get(f'{ds_pfx}_{per}_{s}')
            # Selected event + quality tier: per city×period (ERA5 pipeline, §7)
            row['sel_start']         = r.get(f'{per}_sel_start')
            row['sel_end']           = r.get(f'{per}_sel_end')
            row['sel_duration_days'] = r.get(f'{per}_sel_duration_days')
            row['sel_tier']          = r.get(f'{per}_sel_tier')
            row['validation_status'] = r.get('validation_status')
            _long_rows.append(row)

df_heat_long = (
    pd.DataFrame(_long_rows)
      .sort_values(['cluster25', 'city', 'dataset', 'period'])
      .reset_index(drop=True)
)
df_heat_long.to_csv(_LONG_OUT, index=False)
print(f'  ✓ {_LONG_OUT.name}  ({len(df_heat_long)} rows = '
      f'{df_heat_long["city"].nunique()} cities × 2 datasets × 2 periods, '
      f'{len(df_heat_long.columns)} cols)')

# Preview — exactly the requested fields
_prev = ['cluster25', 'cluster_label', 'city', 'dataset', 'period',
         'n_hot_days', 'n_persistent_events', 'longest_event_days',
         'sel_duration_days', 'sel_tier', 'validation_status']
with pd.option_context('display.max_rows', 24, 'display.width', 220):
    print(df_heat_long[_prev].head(24).to_string(index=False))

---
<a id="nb04-s7c"></a>
### §7c — Heat Event Charts with Error Bars

We draw two figures from `cluster_heat_event_summary.csv`, both with MODIS day (red) and night (blue)
bars per cluster, error bars spanning the cities' range and bar borders coloured by event tier:

| Option | File | Format |
|--------|------|--------|
| 1 — Composite | `heat_event_bars_composite.svg` | All clusters on one A4 |
| 2 — Grid | `heat_event_bars_2x8.svg` | 2 × 8 cluster panels, A4 portrait |
We skip them when both files exist.

> ↑ [Back to Table of Contents](#nb04-toc)

In [ ]:
# ── §7c — Heat event charts with error bars ──────────────────────────────────
import numpy as _np
import matplotlib.pyplot as _plt
import matplotlib.patches as _mpatch
from matplotlib.gridspec import GridSpec as _GS

PUB_DIR = RESULTS_04 / 'publication'
PUB_DIR.mkdir(parents=True, exist_ok=True)

_OUT1 = PUB_DIR / 'heat_event_bars_era5_composite.svg'
_OUT2 = PUB_DIR / 'heat_event_bars_era5_2x8.svg'

_TIER_EC = {'EXCELLENT': '#1a9850', 'GOOD': '#f77f00', 'FAIR': '#999999'}
_DAY_C   = '#d73027'
_NGT_C   = '#4575b4'


def _bar_group(ax, df_cl, day_col, ngt_col, ylabel='Days', title='', bw=0.35):
    _df7  = df_cl.dropna(subset=['cluster25'])
    _sv7  = (_df7.set_index('cluster25')
             [[f'{day_col}_mean', f'{ngt_col}_mean']].fillna(0)
             .max(axis=1).sort_values(ascending=False))
    cids  = _sv7.index.tolist()
    xs   = _np.arange(len(cids))
    for xi, cid in enumerate(cids):
        row = df_cl[df_cl['cluster25'] == cid].iloc[0]
        ec  = _TIER_EC.get(str(row.get('modal_validation_tier', '')), '#cccccc')
        for col, xoff, col_c in [
            (day_col, -bw / 2, _DAY_C),
            (ngt_col, +bw / 2, _NGT_C),
        ]:
            m  = float(row.get(f'{col}_mean', 0) or 0)
            lo = float(row.get(f'{col}_min',  m) or m)
            hi = float(row.get(f'{col}_max',  m) or m)
            ax.bar(xi + xoff, m, width=bw, color=col_c, alpha=0.75,
                   edgecolor=ec, linewidth=1.4, zorder=3)
            ax.errorbar(xi + xoff, m,
                        yerr=[[max(m - lo, 0)], [max(hi - m, 0)]],
                        fmt='none', ecolor='#333333', elinewidth=1.0,
                        capsize=2.5, zorder=4)
    ax.set_xticks(xs)
    ax.set_xticklabels([f'C{int(c):02d}' for c in cids],
                       fontsize=7, rotation=45, ha='right')
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(title, fontsize=8, pad=4)
    ax.grid(axis='y', linestyle=':', alpha=0.45)
    ax.spines[['top', 'right']].set_visible(False)


# ── Option 1: Single A4 composite ────────────────────────────────────────────
if not _OUT1.exists():
    fig1, (ax1a, ax1b) = _plt.subplots(2, 1, figsize=(11.69, 8.27),
                                        gridspec_kw={'hspace': 0.55})
    _bar_group(ax1a, df_cl_heat,
               'day_n_era5_persist_evts', 'night_n_era5_persist_evts',
               ylabel='ERA5 persistent hot events (mean across cities)',
               title='ERA5 Persistent Hot Events per Cluster — Day vs Night')
    _bar_group(ax1b, df_cl_heat,
               'day_era5_longest_days', 'night_era5_longest_days',
               ylabel='Longest event (days, mean)',
               title='ERA5 Longest Persistent Hot Event — Day vs Night')

    _leg = [
        _mpatch.Patch(color=_DAY_C, alpha=0.75, label='Daytime (ERA5)'),
        _mpatch.Patch(color=_NGT_C, alpha=0.75, label='Night-time (ERA5)'),
        _mpatch.Patch(fc='w', ec='#1a9850', lw=1.5, label='EXCELLENT tier'),
        _mpatch.Patch(fc='w', ec='#f77f00', lw=1.5, label='GOOD tier'),
        _mpatch.Patch(fc='w', ec='#999999', lw=1.5, label='FAIR tier'),
    ]
    ax1a.legend(handles=_leg, fontsize=7, loc='upper right',
                framealpha=0.85, ncol=3)

    # National callout
    if 'modis_day_n_hot_days_mean_mean' in df_nat_heat.columns:
        _hd = df_nat_heat.iloc[0]['modis_day_n_hot_days_mean_mean']
        _nc = int(df_nat_heat.iloc[0]['n_cities'])
        ax1a.text(0.01, 0.97,
                  f'National ({_nc} cities)\nMean hot days (day): {_hd:.1f}',
                  transform=ax1a.transAxes, fontsize=7, va='top',
                  bbox=dict(boxstyle='round,pad=0.3', fc='#f5f5f5', ec='#aaa', alpha=0.85))

    fig1.suptitle(
        'Cluster ERA5 Heat Event Summary — 2024\n'
        'Bars = mean across cities in cluster | Error bars = min–max | '
        'Bar border = validation tier',
        fontsize=9, y=0.99)
    fig1.tight_layout(rect=[0, 0, 1, 0.96])
    fig1.savefig(_OUT1, bbox_inches='tight', dpi=config.FIGURE_DPI)
    _plt.close(fig1)
    print(f'  ✓ {_OUT1.name}')
else:
    print(f'  skip {_OUT1.name} (exists)')

# ── Option 2: 2×8 cluster grid ───────────────────────────────────────────────
if not _OUT2.exists():
    _sv_e5  = (df_cl_heat.dropna(subset=['cluster25'])
               .set_index('cluster25')
               [['day_n_era5_persist_evts_mean', 'night_n_era5_persist_evts_mean']]
               .fillna(0).max(axis=1).sort_values(ascending=False))
    _cids   = _sv_e5.index.tolist()
    _nr, _nc = 8, 2
    fig2 = _plt.figure(figsize=(8.27, 11.69))
    _gs  = _GS(_nr, _nc, figure=fig2, hspace=0.70, wspace=0.40,
                top=0.95, bottom=0.03, left=0.09, right=0.97)

    for _idx, cid in enumerate(_cids):
        _ri, _ci = _idx // _nc, _idx % _nc
        ax = fig2.add_subplot(_gs[_ri, _ci])
        _sub = df_cl_heat[df_cl_heat['cluster25'] == cid]
        if _sub.empty:
            ax.set_visible(False)
            continue
        _row = _sub.iloc[0]
        _ec  = _TIER_EC.get(str(_row.get('modal_validation_tier', '')), '#cccccc')

        for _col, _xp, _cc, _lbl in [
            ('day_n_era5_persist_evts',   0, _DAY_C, 'Day (ERA5)'),
            ('night_n_era5_persist_evts', 1, _NGT_C, 'Night (ERA5)'),
        ]:
            _m  = float(_row.get(f'{_col}_mean', 0) or 0)
            _lo = float(_row.get(f'{_col}_min',  _m) or _m)
            _hi = float(_row.get(f'{_col}_max',  _m) or _m)
            ax.bar(_xp, _m, color=_cc, alpha=0.75, edgecolor=_ec, lw=1.0,
                   zorder=3, width=0.6, label=_lbl)
            ax.errorbar(_xp, _m,
                        yerr=[[max(_m - _lo, 0)], [max(_hi - _m, 0)]],
                        fmt='none', ecolor='#333333', elinewidth=0.9,
                        capsize=2, zorder=4)

        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Day', 'Night'], fontsize=5)
        ax.tick_params(axis='y', labelsize=5)
        ax.set_title(f'C{int(cid):02d}  (n={int(_row["n_cities"])})',
                     fontsize=6, pad=2, fontweight='bold')
        if _ci == 0:
            ax.set_ylabel('Hot days', fontsize=5)
        ax.grid(axis='y', linestyle=':', alpha=0.4, linewidth=0.5)
        ax.spines[['top', 'right']].set_visible(False)
        if _idx == 0:
            ax.legend(fontsize=5, loc='upper right', framealpha=0.8,
                      handlelength=0.8, borderpad=0.2, labelspacing=0.1)

    fig2.suptitle(
        'Cluster ERA5 Persistent Events — 2024\n'
        'Error bars = city-level min–max  |  Border = validation tier',
        fontsize=7, y=0.985)
    fig2.savefig(_OUT2, bbox_inches='tight', dpi=config.FIGURE_DPI)
    _plt.close(fig2)
    print(f'  ✓ {_OUT2.name}')
else:
    print(f'  skip {_OUT2.name} (exists)')

print(f'Heat event charts → {PUB_DIR}')

# ── City-level ranked horizontal bar chart ────────────────────────────────────
_OUT3 = PUB_DIR / 'heat_event_bars_era5_city_ranked.svg'
if not _OUT3.exists():
    _RDAY_C  = '#d73027'
    _RNGT_C  = '#4575b4'
    _city_sub = (df_city_heat[['city', 'cluster25',
                                'era5_day_n_persistent_events',
                                'era5_night_n_persistent_events']]
                 .dropna(subset=['era5_day_n_persistent_events'])
                 .sort_values('era5_day_n_persistent_events', ascending=True))
    _cmap_cl  = {cid: _plt.cm.tab20(i / 20)
                 for i, cid in enumerate(sorted(_city_sub['cluster25'].dropna().unique()))}
    fig3, ax3 = _plt.subplots(figsize=(8.27, max(6.0, len(_city_sub) * 0.22)))
    _ys = _np.arange(len(_city_sub))
    # Day bars (full height, red) and night bars (half height, blue) side by side
    ax3.barh(_ys + 0.2, _city_sub['era5_day_n_persistent_events'],   height=0.38,
             color=_RDAY_C, alpha=0.78, zorder=3, label='Daytime (ERA5)')
    ax3.barh(_ys - 0.2, _city_sub['era5_night_n_persistent_events'], height=0.38,
             color=_RNGT_C, alpha=0.78, zorder=3, label='Nighttime (ERA5)')
    ax3.set_yticks(_ys)
    ax3.set_yticklabels(_city_sub['city'], fontsize=5.5)
    ax3.set_xlabel('ERA5 Persistent Hot Events (2024)', fontsize=9)
    ax3.set_title(
        'Cities Ranked by ERA5 Daytime Persistent Hot Events — 2024\n'
        'Red = Daytime  |  Blue = Nighttime  |  N events above P90 threshold',
        fontsize=9, pad=6)
    ax3.grid(axis='x', linestyle=':', alpha=0.45)
    ax3.spines[['top', 'right']].set_visible(False)
    ax3.legend(fontsize=7, loc='lower right', framealpha=0.85)
    fig3.tight_layout()
    fig3.savefig(_OUT3, bbox_inches='tight', dpi=config.FIGURE_DPI)
    _plt.close(fig3)
    print(f'  ✓ {_OUT3.name}')
else:
    print(f'  skip {_OUT3.name} (exists)')

# ── §7c  MODIS variants — removed (MODIS is validation-only) ───────────────────
print('  §7c MODIS charts — skipped (MODIS validation-only, not included in analysis)')

# ── §7c  Combined variants ────────────────────────────────────────────────────
_OUT1C = PUB_DIR / 'heat_event_bars_combined_composite.svg'
_OUT2C = PUB_DIR / 'heat_event_bars_combined_2x8.svg'
_OUT3C = PUB_DIR / 'heat_event_bars_combined_city_ranked.svg'
_MDAY_C7 = '#fc8d59'
_MNGT_C7 = '#74add1'

if not _OUT1C.exists():
    fig1c, (axs1ca, axs1cb) = _plt.subplots(1, 2, figsize=(11.69, 5.85),
                                              gridspec_kw={'wspace': 0.35})
    _bar_group(axs1ca, df_cl_heat,
               'day_n_era5_persist_evts', 'night_n_era5_persist_evts',
               ylabel='Events', title='ERA5 Persistent Events')
    _bar_group(axs1cb, df_cl_heat,
               'day_era5_longest_days', 'night_era5_longest_days',
               ylabel='Days', title='ERA5 Longest Event')
    _legc = [
        _mpatch.Patch(color=_DAY_C, alpha=0.75, label='Daytime'),
        _mpatch.Patch(color=_NGT_C, alpha=0.75, label='Nighttime'),
        _mpatch.Patch(fc='w', ec='#1a9850', lw=1.5, label='EXCELLENT tier'),
        _mpatch.Patch(fc='w', ec='#f77f00', lw=1.5, label='GOOD tier'),
        _mpatch.Patch(fc='w', ec='#999999', lw=1.5, label='FAIR tier'),
    ]
    axs1ca.legend(handles=_legc, fontsize=7, loc='upper right', framealpha=0.85, ncol=2)
    fig1c.suptitle(
        'Cluster ERA5 Heat Event Summary — 2024\n'
        'Error bars = city-level min–max  |  Border = validation tier',
        fontsize=9, y=1.02)
    fig1c.tight_layout(rect=[0, 0, 1, 0.96])
    fig1c.savefig(_OUT1C, bbox_inches='tight', dpi=config.FIGURE_DPI)
    _plt.close(fig1c)
    print(f'  ✓ {_OUT1C.name}')
else:
    print(f'  skip {_OUT1C.name} (exists)')

# combined 2×8 removed — combined ERA5+MODIS metric charts excluded
print('  §7c combined 2×8 — skipped (removed)')

if not _OUT3C.exists():
    _city_ord = (df_city_heat[['city', 'era5_day_n_persistent_events',
                                'era5_night_n_persistent_events',
                                'modis_day_n_hot_days', 'modis_night_n_hot_days']]
                 .dropna(subset=['era5_day_n_persistent_events'])
                 .sort_values('era5_day_n_persistent_events', ascending=True))
    _n_co = len(_city_ord)
    fig3c, (ax3ca, ax3cb) = _plt.subplots(
        1, 2, figsize=(11.69, max(6.0, _n_co * 0.22)),
        gridspec_kw={'wspace': 0.05})
    _ys_co = _np.arange(_n_co)
    ax3ca.barh(_ys_co + 0.2, _city_ord['era5_day_n_persistent_events'],
               height=0.38, color=_DAY_C, alpha=0.78, zorder=3, label='Day (ERA5)')
    ax3ca.barh(_ys_co - 0.2, _city_ord['era5_night_n_persistent_events'],
               height=0.38, color=_NGT_C, alpha=0.78, zorder=3, label='Night (ERA5)')
    ax3ca.set_yticks(_ys_co)
    ax3ca.set_yticklabels(_city_ord['city'], fontsize=5.5)
    ax3ca.set_xlabel('ERA5 Persistent Events', fontsize=8)
    ax3ca.set_title('ERA5 Persistent Hot Events', fontsize=9, fontweight='bold')
    ax3ca.grid(axis='x', linestyle=':', alpha=0.45)
    ax3ca.spines[['top', 'right']].set_visible(False)
    ax3ca.legend(fontsize=7, loc='lower right', framealpha=0.85)
    ax3cb.barh(_ys_co + 0.2, _city_ord['modis_day_n_hot_days'],
               height=0.38, color=_DAY_C, alpha=0.78, zorder=3, label='Day (MODIS)')
    ax3cb.barh(_ys_co - 0.2, _city_ord['modis_night_n_hot_days'],
               height=0.38, color=_NGT_C, alpha=0.78, zorder=3, label='Night (MODIS)')
    ax3cb.set_yticks(_ys_co)
    ax3cb.set_yticklabels([], fontsize=5.5)
    ax3cb.set_xlabel('MODIS Hot Days', fontsize=8)
    ax3cb.set_title('MODIS Hot Days', fontsize=9, fontweight='bold')
    ax3cb.grid(axis='x', linestyle=':', alpha=0.45)
    ax3cb.spines[['top', 'right']].set_visible(False)
    ax3cb.legend(fontsize=7, loc='lower right', framealpha=0.85)
    fig3c.suptitle(
        'Cities: ERA5 Persistent Events (left) vs MODIS Hot Days (right) — 2024\n'
        'Same city order; sorted by ERA5 daytime persistent events',
        fontsize=9, y=1.00)
    fig3c.tight_layout(rect=[0, 0, 1, 0.96])
    fig3c.savefig(_OUT3C, bbox_inches='tight', dpi=config.FIGURE_DPI)
    _plt.close(fig3c)
    print(f'  ✓ {_OUT3C.name}')
else:
    print(f'  skip {_OUT3C.name} (exists)')


<a id="nb04-s8"></a>
## §8 — Daily LST Time-Series Charts

We draw a two-panel chart per city (MODIS above, ERA5 below, A4 landscape) with the P90 threshold
(dashed), every persistent event shaded grey and the selected event bracketed. We show the first city
here and save the rest.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── Per-city daily LST time-series charts ────────────────────────────────────
CHARTS_CITY_DIR = CHARTS_DIR / 'city_daily_timeseries'
CHARTS_CITY_DIR.mkdir(parents=True, exist_ok=True)

_first_shown = False

for city in CITIES:
    modis_csv = RAW_DIR / f'{city}_MODIS_UC_daily_{STUDY_YEAR}.csv'
    era5_csv  = RAW_DIR / f'{city}_ERA5_UC_daily_{STUDY_YEAR}.csv'

    # Load full DataFrames (with min/median/max columns)
    modis_full = pd.read_csv(modis_csv) if modis_csv.exists() else None
    era5_full  = pd.read_csv(era5_csv)  if era5_csv.exists()  else None

    if modis_full is None and era5_full is None:
        print(f"  SKIP {city}: no data")
        continue

    out_svg = CHARTS_CITY_DIR / f'{city}_daily_LST_{STUDY_YEAR}.svg'

    show  = not _first_shown   # show first city inline
    fig = create_nb04_event_plot(
        modis_df      = modis_full,
        era5_df       = era5_full,
        city_name     = city,
        study_year    = STUDY_YEAR,
        audit_day     = _audit_day.get(city, {}),
        audit_night   = _audit_night.get(city, {}),
        output_path   = str(out_svg),
        show_inline   = show,
    )

    if not _first_shown and fig is not None:
        _first_shown = True
        print(f"  ↑ {city}: displayed inline")
    elif fig is not None:
        print(f"  ✓ {city}: saved -> {out_svg.name}")

print(f"\nAll city daily charts -> {CHARTS_CITY_DIR}")


<a id="nb04-s8a"></a>
### §8a — Cluster Composite Daily Charts

We average the city series of each cluster per date, draw the cluster's daily chart, and draw the event
frequency chart of all clusters.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# -- Cluster composite daily series
CHARTS_CL_DIR = CHARTS_DIR / 'cluster_daily_composites'
CHARTS_CL_DIR.mkdir(parents=True, exist_ok=True)

# Build city-level full DataFrames
_city_modis_full = {}
_city_era5_full  = {}
for city in CITIES:
    mcsvp = RAW_DIR / f'{city}_MODIS_UC_daily_{STUDY_YEAR}.csv'
    ecsvp = RAW_DIR / f'{city}_ERA5_UC_daily_{STUDY_YEAR}.csv'
    if mcsvp.exists():
        _df = pd.read_csv(mcsvp)
        _df['date'] = pd.to_datetime(_df['date'], errors='coerce')
        _city_modis_full[city] = _df
    if ecsvp.exists():
        _df = pd.read_csv(ecsvp)
        _df['date'] = pd.to_datetime(_df['date'], errors='coerce')
        _city_era5_full[city] = _df

# Build cluster composites (mean across cities per date)
cl_modis = build_cluster_daily_series(_city_modis_full, _city_to_cluster)
cl_era5  = build_cluster_daily_series(_city_era5_full,  _city_to_cluster)

# Save cluster composite CSVs
for cnum, df_cl in cl_modis.items():
    df_cl.to_csv(CLUSTER_DIR / f'Cluster{cnum:02d}_MODIS_composite_{STUDY_YEAR}.csv', index=False)
for cnum, df_cl in cl_era5.items():
    df_cl.to_csv(CLUSTER_DIR / f'Cluster{cnum:02d}_ERA5_composite_{STUDY_YEAR}.csv', index=False)
print(f'  ok Cluster composite CSVs -> {CLUSTER_DIR}')

# Cluster event frequency barplots
fig_ef_d = create_cluster_event_frequency_chart(
    cluster_event_summary = df_cl_events_day,
    city_event_records    = _audit_day,
    cluster_lookup        = _city_to_cluster,
    period                = 'day',
    output_path           = str(CHARTS_DIR / 'Cluster_event_frequency_day.svg'),
    dpi                   = config.FIGURE_DPI,
)
fig_ef_n = create_cluster_event_frequency_chart(
    cluster_event_summary = df_cl_events_night,
    city_event_records    = _audit_night,
    cluster_lookup        = _city_to_cluster,
    period                = 'night',
    output_path           = str(CHARTS_DIR / 'Cluster_event_frequency_night.svg'),
    dpi                   = config.FIGURE_DPI,
)
print('  ok Cluster event frequency charts (day + night).')

def _extract_period_df(composite_df, period):
    # Extract period-specific sub-DF, rename {period}_lst_median to lst_median
    if composite_df is None:
        return None
    col = f'{period}_lst_median'
    if col not in composite_df.columns:
        return None
    df = composite_df[['date', col]].copy()
    df = df.rename(columns={col: 'lst_median'})
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date', 'lst_median']).sort_values('date').reset_index(drop=True)
    return df if not df.empty else None

# Plot cluster composite daily charts with real event audits
_first_cl = True
for cnum in sorted(set(cl_era5.keys()) | set(cl_modis.keys())):
    edf_cl = cl_era5.get(cnum)
    mdf_cl = cl_modis.get(cnum)
    if edf_cl is None and mdf_cl is None:
        continue

    # Compute real cluster-level audits from the composite series
    _cl_audit_d = run_city_event_pipeline(
        modis_df        = _extract_period_df(mdf_cl, 'day'),
        era5_df         = _extract_period_df(edf_cl, 'day'),
        city            = f'Cluster {cnum:02d}',
        period          = 'day',
        percentile      = HOT_PERCENTILE,
        min_consecutive = PERSISTENT_THRESHOLD,
    )
    _cl_audit_n = run_city_event_pipeline(
        modis_df        = _extract_period_df(mdf_cl, 'night'),
        era5_df         = _extract_period_df(edf_cl, 'night'),
        city            = f'Cluster {cnum:02d}',
        period          = 'night',
        percentile      = HOT_PERCENTILE,
        min_consecutive = PERSISTENT_THRESHOLD,
    )

    out_svg_cl = CHARTS_CL_DIR / f'Cluster{cnum:02d}_daily_LST_{STUDY_YEAR}.svg'
    fig_cl = create_nb04_event_plot(
        modis_df    = mdf_cl,
        era5_df     = edf_cl,
        city_name   = f'Cluster {cnum:02d} Composite',
        study_year  = STUDY_YEAR,
        audit_day   = _cl_audit_d,
        audit_night = _cl_audit_n,
        output_path = str(out_svg_cl),
        show_inline = _first_cl,
    )
    if _first_cl and fig_cl is not None:
        _first_cl = False
        print(f'  inline: Cluster {cnum:02d} displayed')

print(f'  ok Cluster composite charts -> {CHARTS_CL_DIR}')

---
<a id="nb04-s8b"></a>
### §8b — Publication 2 × 8 Cluster Daily Composite

We draw the 16 clusters on one A4 page (8 rows × 2): the city lines faint, the cluster median day (red)
and night (blue) in bold, and the P90 thresholds as thin dashed lines. The P90 values, selected events
and correlations go to `publication/cluster_daily_annotation_table.csv` instead of the panels. We skip
it when both files exist.

> ↑ [Back to Table of Contents](#nb04-toc)

In [ ]:
# ── §8b — Publication 2×8 cluster daily composite ────────────────────────────
import matplotlib.pyplot as _plt2
import matplotlib.dates as _mdates
from matplotlib.gridspec import GridSpec as _GS2

PUB_DIR = RESULTS_04 / 'publication'
PUB_DIR.mkdir(parents=True, exist_ok=True)

_OUT_GRID = PUB_DIR / 'cluster_daily_composite_2x8.svg'
_OUT_ANN  = PUB_DIR / 'cluster_daily_annotation_table.csv'

if _OUT_GRID.exists() and _OUT_ANN.exists():
    print('§8b cache hit — publication composite already exists')
else:
    _DAY_C  = '#d73027'
    _NGT_C  = '#4575b4'
    # Multi-city clusters only (≥2 cities) — single-city excluded from composite
    _MULTI_CITY = [1, 4, 7, 9, 11, 12, 13, 16, 25]
    _cids   = sorted(c for c in (set(cl_modis.keys()) | set(cl_era5.keys()))
                     if c in _MULTI_CITY)
    _nr, _nc = 5, 2   # 9 clusters → 5×2

    fig = _plt2.figure(figsize=(8.27, 11.69))
    _gs = _GS2(_nr, _nc, figure=fig,
                hspace=0.55, wspace=0.28,
                top=0.96, bottom=0.03, left=0.09, right=0.97)

    for _idx, cid in enumerate(_cids):
        _ri, _ci = _idx // _nc, _idx % _nc
        ax = fig.add_subplot(_gs[_ri, _ci])

        # ── cluster-level composite DFs ───────────────────────────────────────
        mdf = cl_modis.get(cid)
        if mdf is not None:
            mdf = mdf.copy()
            mdf['date'] = pd.to_datetime(mdf['date'], errors='coerce')
            mdf = mdf.sort_values('date')

        # ── individual city ghost lines ───────────────────────────────────────
        _cl_cities = [c for c, cn in _city_to_cluster.items() if cn == cid]
        for _city in _cl_cities:
            _cdf = _city_modis_full.get(_city)
            if _cdf is None:
                continue
            _cdf = _cdf.copy()
            _cdf['date'] = pd.to_datetime(_cdf['date'], errors='coerce')
            _cdf = _cdf.sort_values('date').dropna(subset=['date'])
            if 'day_lst_median' in _cdf.columns:
                ax.plot(_cdf['date'], _cdf['day_lst_median'],
                        color=_DAY_C, lw=0.35, alpha=0.15, zorder=1)
            if 'night_lst_median' in _cdf.columns:
                ax.plot(_cdf['date'], _cdf['night_lst_median'],
                        color=_NGT_C, lw=0.35, alpha=0.15, zorder=1)

        # ── cluster median (bold) ─────────────────────────────────────────────
        if mdf is not None:
            if 'day_lst_median' in mdf.columns:
                ax.plot(mdf['date'], mdf['day_lst_median'],
                        color=_DAY_C, lw=1.4, zorder=3,
                        label='Day' if _idx == 0 else '')
            if 'night_lst_median' in mdf.columns:
                ax.plot(mdf['date'], mdf['night_lst_median'],
                        color=_NGT_C, lw=1.4, zorder=3,
                        label='Night' if _idx == 0 else '')

        # ── P90 dashed threshold lines (no label text) ────────────────────────
        if 'df_cl_heat' in dir() or 'df_cl_heat' in globals():
            _hrow = df_cl_heat[df_cl_heat['cluster25'] == cid]
            if not _hrow.empty:
                _r = _hrow.iloc[0]
                _p90d = _r.get('modis_day_p90_threshold_C_mean')
                _p90n = _r.get('modis_night_p90_threshold_C_mean')
                if pd.notna(_p90d):
                    ax.axhline(_p90d, color=_DAY_C, ls='--', lw=0.65, alpha=0.55, zorder=2)
                if pd.notna(_p90n):
                    ax.axhline(_p90n, color=_NGT_C, ls='--', lw=0.65, alpha=0.55, zorder=2)

        # ── axes formatting ───────────────────────────────────────────────────
        ax.xaxis.set_major_locator(_mdates.MonthLocator(interval=2))
        ax.xaxis.set_major_formatter(_mdates.DateFormatter('%b'))
        ax.tick_params(axis='x', labelsize=5, rotation=30)
        ax.tick_params(axis='y', labelsize=5)
        ax.set_title(f'C{int(cid):02d}  (n={len(_cl_cities)})',
                     fontsize=6, pad=2, fontweight='bold')
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='y', linestyle=':', lw=0.4, alpha=0.4)
        if _ci == 0:
            ax.set_ylabel('LST (°C)', fontsize=5)

    # Legend on first panel
    fig.axes[0].legend(fontsize=5, loc='upper right', framealpha=0.8,
                       handlelength=0.9, borderpad=0.3, labelspacing=0.15)

    fig.suptitle(
        'Cluster Composite Daily LST — 2024  (MODIS UC)'
        'Ghost lines = individual cities | Bold = cluster median | '
        'Dashed = P90 threshold',
        fontsize=6.5, y=0.995)
    fig.savefig(_OUT_GRID, bbox_inches='tight', dpi=config.FIGURE_DPI)
    _plt2.close(fig)
    print(f'  ✓ {_OUT_GRID.name}')

    # ── Annotation table (replaces in-chart text boxes) ──────────────────────
    # Pulls annotation-relevant metrics from df_cl_heat (if available)
    _ann_rows = []
    for cid in sorted(set(cl_modis.keys()) | set(cl_era5.keys())):
        _cl_cities = [c for c, cn in _city_to_cluster.items() if cn == cid]
        _ar = {
            'cluster25': int(cid),
            'n_cities':  len(_cl_cities),
            'city_names': ', '.join(sorted(_cl_cities)),
        }
        if 'df_cl_heat' in dir() or 'df_cl_heat' in globals():
            _hrow = df_cl_heat[df_cl_heat['cluster25'] == cid]
            if not _hrow.empty:
                _r = _hrow.iloc[0]
                for _col in [
                    'modal_validation_tier',
                    'modis_day_p90_threshold_C_mean',
                    'modis_night_p90_threshold_C_mean',
                    'era5_day_p90_threshold_C_mean',
                    'era5_night_p90_threshold_C_mean',
                    'modis_day_n_hot_days_mean',
                    'modis_night_n_hot_days_mean',
                    'modis_day_n_persistent_events_mean',
                    'modis_night_n_persistent_events_mean',
                    'day_corr_r_mean', 'night_corr_r_mean',
                    'day_corr_r2_mean', 'night_corr_r2_mean',
                    'day_corr_bias_mean', 'night_corr_bias_mean',
                    'day_corr_rmse_mean', 'night_corr_rmse_mean',
                    'day_sel_start', 'day_sel_end',
                    'day_sel_duration_days', 'day_sel_tier',
                    'night_sel_start', 'night_sel_end',
                    'night_sel_duration_days', 'night_sel_tier',
                ]:
                    if _col in _r.index:
                        _ar[_col] = _r[_col]
        _ann_rows.append(_ar)
    pd.DataFrame(_ann_rows).to_csv(_OUT_ANN, index=False)
    print(f'  ✓ {_OUT_ANN.name}')
    print(f'  → all annotation data for in-chart labels exported to companion table')
    print(f'  Charts → {PUB_DIR}')

---
<a id="nb04-s8c"></a>
### §8c — National Cluster Heat Event Violin

We compare the MODIS hot days of the nine clusters with at least two cities (the seven single-city
clusters are left out), sorted by mean day hot days:

| Side | Colour | Metric |
|------|--------|--------|
| Right | Red-orange | MODIS day hot days — city spread within cluster |
| Left  | Blue       | MODIS night hot days |
The thick tick marks the median across cities. **Output:** `publication/national_cluster_heat_violin.svg`

> ↑ [Back to Table of Contents](#nb04-toc)

In [ ]:
# ── §8c — National cluster heat event violin (multi-city clusters only) ───────
import numpy as _npv
from scipy.stats import gaussian_kde as _gkde
import matplotlib.pyplot as _pltv
import matplotlib.ticker  as _mtv
import matplotlib.lines   as _mlv
import matplotlib.patches as _mpv

PUB_DIR = RESULTS_04 / 'publication'
PUB_DIR.mkdir(parents=True, exist_ok=True)
_OUT_8C = PUB_DIR / 'national_cluster_heat_violin_era5.svg'

if _OUT_8C.exists():
    print(f'§8c cache hit — {_OUT_8C.name}')
else:
    _MULTI = [4, 7, 9, 11, 13, 25]
    _vc = (df_city_heat[['city', 'cluster25',
                          'era5_day_n_persistent_events',
                          'era5_night_n_persistent_events']]
           .dropna(subset=['cluster25'])
           .copy())
    _vc['cluster25'] = _vc['cluster25'].astype(int)
    _vc = _vc[_vc['cluster25'].isin(_MULTI)]   # multi-city only

    # Sort clusters by descending mean daytime ERA5 persistent events
    _cl_means = (_vc.groupby('cluster25')[['era5_day_n_persistent_events',
                                           'era5_night_n_persistent_events']]
                 .mean().max(axis=1).sort_values(ascending=False))
    _sorted   = _cl_means.index.tolist()
    _n        = len(_sorted)
    _cy       = {c: i + 1 for i, c in enumerate(_sorted)}

    _DAY_C = '#d73027'
    _NGT_C = '#4575b4'
    _HW    = 0.38
    _KPT   = 300

    fig_v, ax_v = _pltv.subplots(figsize=(11.0, max(4.0, _n * 0.55)))

    def _hv(data, y0, side, col):
        arr = _npv.asarray(data, dtype=float)
        arr = arr[~_npv.isnan(arr)]
        if len(arr) < 2:
            if len(arr) == 1:
                off = 0.10 if side == 'right' else -0.10
                ax_v.scatter(arr[0], y0 + off, color=col, s=60, zorder=7, linewidths=0)
            return
        xlo = max(arr.min() - 0.5, 0)
        xhi = arr.max() + 0.5
        xg  = _npv.linspace(xlo, xhi, _KPT)
        try:
            sig = _npv.std(arr)
            bw  = max(0.5, sig * 0.4)
            kde = _gkde(arr, bw_method=bw / (sig if sig > 0 else 1))
            dn  = kde(xg)
        except Exception:
            return
        dn = dn / dn.max() * _HW if dn.max() > 0 else dn
        yl = _npv.full_like(xg, y0) if side == 'right' else y0 - dn
        yh = y0 + dn if side == 'right' else _npv.full_like(xg, y0)
        ax_v.fill_between(xg, yl, yh, color=col, alpha=0.45, linewidth=0)
        ax_v.plot(xg, yh, color=col, lw=0.9, alpha=0.85)
        ax_v.plot(xg, yl, color=col, lw=0.9, alpha=0.85)
        med  = float(_npv.median(arr))
        medn = float(_npv.interp(med, xg, dn))
        if side == 'right':
            ax_v.plot([med, med], [y0, y0 + medn],
                      color=col, lw=2.5, solid_capstyle='round', zorder=5)
        else:
            ax_v.plot([med, med], [y0 - medn, y0],
                      color=col, lw=2.5, solid_capstyle='round', zorder=5)

    for _cid in _sorted:
        _y0  = _cy[_cid]
        _sub = _vc[_vc['cluster25'] == _cid]
        _hv(_sub['era5_day_n_persistent_events'].dropna().values,   _y0, 'right', _DAY_C)
        _hv(_sub['era5_night_n_persistent_events'].dropna().values, _y0, 'left',  _NGT_C)

    _xmax = max(df_city_heat['era5_day_n_persistent_events'].dropna().max(),
                df_city_heat['era5_night_n_persistent_events'].dropna().max()) + 2

    ax_v.set_xlim(0, _xmax)
    ax_v.set_ylim(_n + 0.7, 0.3)
    ax_v.set_xlabel('ERA5 Persistent Hot Events (2024)', fontsize=11)
    ax_v.xaxis.set_major_locator(_mtv.MultipleLocator(5))
    ax_v.xaxis.set_minor_locator(_mtv.MultipleLocator(1))
    ax_v.grid(axis='x', which='major', ls='--', alpha=0.45)
    ax_v.grid(axis='x', which='minor', ls=':',  alpha=0.25)
    ax_v.set_yticks(list(range(1, _n + 1)))
    _ysd_lbl = []
    for _c8 in _sorted:
        _sub8  = _vc[_vc['cluster25'] == _c8]
        _sd8d  = float(_npv.std(_sub8['era5_day_n_persistent_events'].dropna().values)) if len(_sub8) >= 2 else float('nan')
        _sd8n  = float(_npv.std(_sub8['era5_night_n_persistent_events'].dropna().values)) if len(_sub8) >= 2 else float('nan')
        _sd8s  = (f'  σ_d={_sd8d:.1f}\nσ_n={_sd8n:.1f}'
                  if not (_npv.isnan(_sd8d) or _npv.isnan(_sd8n)) else '')
        _ysd_lbl.append(f'C{int(_c8):02d}{_sd8s}')
    ax_v.set_yticklabels(_ysd_lbl, fontsize=9)
    ax_v.tick_params(axis='y', length=0)
    ax_v.spines[['top', 'right']].set_visible(False)
    ax_v.legend(handles=[
        _mpv.Patch(color=_DAY_C, alpha=0.65, label='Day persistent events (ERA5)'),
        _mpv.Patch(color=_NGT_C, alpha=0.65, label='Night persistent events (ERA5)'),
        _mlv.Line2D([], [], color='#333', lw=2.5, label='Median'),
    ], fontsize=8, framealpha=0.85, loc='lower right')
    ax_v.set_title(
        'Multi-City Clusters — ERA5 Persistent Hot Events (2024)\n'
        'Sorted by mean daytime events; '
        'half-violin = city spread within cluster  |  ≥3 cities',
        fontsize=10, fontweight='bold', pad=8)

    fig_v.savefig(_OUT_8C, bbox_inches='tight', dpi=config.FIGURE_DPI)
    _pltv.close(fig_v)
    print(f'  ✓ {_OUT_8C.name}  ({_n} multi-city clusters)')

# ── §8c  MODIS + combined national violin — removed (MODIS validation-only) ───────
print('  §8c MODIS/combined national violin — skipped (removed)')

---
<a id="nb04-s8d"></a>
### §8d — Cluster Heat Event Violin Grid (5 × 2)

We show the spread of the cities' MODIS hot days within each of the nine multi-city clusters (5 × 2 on
A4): day (orange-red) beside night (blue), the median as a bar, the points shown for small samples.

**Output:** `publication/cluster_heat_violin_5x2.svg`

> ↑ [Back to Table of Contents](#nb04-toc)

In [ ]:
# ── §8d — Cluster 5×2 heat event violin grid (multi-city only) ───────────────
import numpy as _npg
from scipy.stats import gaussian_kde as _gkdeg
import matplotlib.pyplot as _pltg
from matplotlib.gridspec import GridSpec as _GSg

PUB_DIR = RESULTS_04 / 'publication'
_OUT_8D = PUB_DIR / 'cluster_heat_violin_era5_3x2.svg'

if _OUT_8D.exists():
    print(f'§8d cache hit — {_OUT_8D.name}')
else:
    _MULTI = [4, 7, 9, 11, 13, 25]
    _vc2 = (df_city_heat[['city', 'cluster25',
                           'era5_day_n_persistent_events',
                           'era5_night_n_persistent_events']]
            .dropna(subset=['cluster25'])
            .copy())
    _vc2['cluster25'] = _vc2['cluster25'].astype(int)
    _vc2 = _vc2[_vc2['cluster25'].isin(_MULTI)]

    _sv_g   = (_vc2.groupby('cluster25')[['era5_day_n_persistent_events',
                                          'era5_night_n_persistent_events']]
               .mean().max(axis=1).sort_values(ascending=False))
    _cids_g = _sv_g.index.tolist()
    _NR, _NC = 3, 2
    _DAY_C   = '#d73027'
    _NGT_C   = '#4575b4'

    fig_g = _pltg.figure(figsize=(8.27, 11.69))
    _gsg  = _GSg(_NR, _NC, figure=fig_g,
                  hspace=0.70, wspace=0.35,
                  top=0.94, bottom=0.04, left=0.10, right=0.97)

    def _mini_vln(ax, vals, xpos, col, width=0.35):
        arr = _npg.asarray(vals, dtype=float)
        arr = arr[~_npg.isnan(arr)]
        if len(arr) == 0:
            return
        if len(arr) == 1:
            ax.scatter(xpos, arr[0], color=col, s=50, zorder=5, linewidths=0)
            return
        if len(arr) < 4:
            for v in arr:
                ax.scatter(xpos, v, color=col, s=22, alpha=0.65, zorder=5, linewidths=0)
            ax.plot([xpos, xpos], [float(_npg.min(arr)), float(_npg.max(arr))],
                    color=col, lw=1.2, zorder=4, alpha=0.7)
            return
        try:
            sig = _npg.std(arr)
            bw  = max(0.5, sig * 0.4)
            kde = _gkdeg(arr, bw_method=bw / (sig if sig > 0 else 1))
            yg  = _npg.linspace(arr.min() - 0.5, arr.max() + 0.5, 200)
            dn  = kde(yg)
            dn  = dn / dn.max() * width if dn.max() > 0 else dn
            ax.fill_betweenx(yg, xpos - dn / 2, xpos + dn / 2,
                             color=col, alpha=0.45, linewidth=0)
            ax.plot(xpos - dn / 2, yg, color=col, lw=0.8, alpha=0.8)
            ax.plot(xpos + dn / 2, yg, color=col, lw=0.8, alpha=0.8)
            med   = float(_npg.median(arr))
            med_d = float(_npg.interp(med, yg, dn))
            ax.plot([xpos - med_d / 2, xpos + med_d / 2], [med, med],
                    color=col, lw=2.0, zorder=6)
        except Exception:
            for v in arr:
                ax.scatter(xpos, v, color=col, s=15, alpha=0.6, zorder=5, linewidths=0)

    for _idx, cid in enumerate(_cids_g):
        _ri, _ci = _idx // _NC, _idx % _NC
        ax = fig_g.add_subplot(_gsg[_ri, _ci])
        _sub = _vc2[_vc2['cluster25'] == cid]
        _n_c = len(_sub)

        _mini_vln(ax, _sub['era5_day_n_persistent_events'].values,   0.75, _DAY_C)
        _mini_vln(ax, _sub['era5_night_n_persistent_events'].values, 1.25, _NGT_C)

        ax.set_xticks([0.75, 1.25])
        ax.set_xticklabels(['Day', 'Night'], fontsize=7)
        ax.set_xlim(0.3, 1.7)
        ax.tick_params(axis='y', labelsize=7)
        _sd_d8  = float(_npg.std(_sub['era5_day_n_persistent_events'].dropna().values)) if _n_c >= 2 else float('nan')
        _sd_n8  = float(_npg.std(_sub['era5_night_n_persistent_events'].dropna().values)) if _n_c >= 2 else float('nan')
        _sd_t8  = (f'  σ_d={_sd_d8:.1f}, σ_n={_sd_n8:.1f} days'
                   if not (_npg.isnan(_sd_d8) or _npg.isnan(_sd_n8)) else '')
        ax.set_title(f'Cluster {int(cid):02d}  (n={_n_c} cities){_sd_t8}',
                     fontsize=7, pad=3, fontweight='bold')
        if _ci == 0:
            ax.set_ylabel('ERA5 Persistent Events', fontsize=7)
        ax.grid(axis='y', linestyle=':', lw=0.5, alpha=0.45)
        ax.spines[['top', 'right']].set_visible(False)
        if _idx == 0:
            import matplotlib.patches as _mp_leg
            _lh = [_mp_leg.Patch(color=_DAY_C, alpha=0.65, label='Day (ERA5)'),
                   _mp_leg.Patch(color=_NGT_C, alpha=0.65, label='Night (ERA5)')]
            ax.legend(handles=_lh, fontsize=6.5, loc='upper right',
                      framealpha=0.85, handlelength=1.0)

    # Hide unused panels
    for _idx in range(len(_cids_g), _NR * _NC):
        _ri, _ci = _idx // _NC, _idx % _NC
        fig_g.add_subplot(_gsg[_ri, _ci]).set_visible(False)

    fig_g.suptitle(
        'Multi-City Cluster ERA5 Persistent Hot Events — 2024\n'
        'Violin width ∝ density across cities | Bar = median  |  ≥3 cities',
        fontsize=8, y=0.99)
    fig_g.savefig(_OUT_8D, bbox_inches='tight', dpi=config.FIGURE_DPI)
    _pltg.close(fig_g)
    print(f'  ✓ {_OUT_8D.name}  ({len(_cids_g)} clusters, {_NR}×{_NC} grid)')

# ── §8d  MODIS + combined cluster violin — removed (MODIS validation-only) ─────────
print('  §8d MODIS/combined violin — skipped (removed)')

---
<a id="nb04-s8e"></a>
### §8e — Cluster Butterfly Heat Event Violin Grid (3 × 2)

We draw a butterfly violin per cluster, day hot days on the right (red) against night on the left
(blue), for the six clusters with at least three cities (3 × 2 on A4), with σ_d and σ_n in each title.

> ↑ [Back to Table of Contents](#nb04-toc)

In [ ]:
# ── §8e — Cluster butterfly heat event violin grid (3×2, ≥3 cities) ──────────
import numpy as _npe
from scipy.stats import gaussian_kde as _gkde_e
import matplotlib.pyplot as _plte
import matplotlib.patches as _mpe
from matplotlib.gridspec import GridSpec as _GSe

PUB_DIR  = RESULTS_04 / 'publication'
_OUT_8E  = PUB_DIR / 'cluster_heat_butterfly_era5_3x2.svg'

if _OUT_8E.exists():
    print(f'§8e cache hit — {_OUT_8E.name}')
else:
    _MULTI3 = [4, 7, 9, 11, 13, 25]
    _vc3 = (df_city_heat[['city', 'cluster25',
                           'era5_day_n_persistent_events',
                           'era5_night_n_persistent_events']]
            .dropna(subset=['cluster25'])
            .copy())
    _vc3['cluster25'] = _vc3['cluster25'].astype(int)
    _vc3 = _vc3[_vc3['cluster25'].isin(_MULTI3)]
    _sv_e   = (_vc3.groupby('cluster25')[['era5_day_n_persistent_events',
                                          'era5_night_n_persistent_events']]
               .mean().max(axis=1).sort_values(ascending=False))
    _cids_e = _sv_e.index.tolist()

    _NRE, _NCE = 3, 2
    _DAY_E = '#d73027'
    _NGT_E = '#4575b4'

    fig_e = _plte.figure(figsize=(8.27, 11.69))
    _gse  = _GSe(_NRE, _NCE, figure=fig_e,
                  hspace=0.72, wspace=0.42,
                  top=0.93, bottom=0.04, left=0.12, right=0.96)
    fig_e.suptitle(
        'Cluster ERA5 Persistent Events — Day (right ▶) vs Night (left ◀)\n'
        'N persistent ERA5 events per city  |  ≥3 cities per cluster',
        fontsize=8, fontweight='bold')

    _leg_e_d = _mpe.Patch(facecolor=_DAY_E, alpha=0.6, label='Daytime')
    _leg_e_n = _mpe.Patch(facecolor=_NGT_E, alpha=0.6, label='Nighttime')

    def _draw_half_e(ax, vals, xpos, side, color, half_w=0.35):
        arr = _npe.asarray(vals, float); arr = arr[~_npe.isnan(arr)]
        if len(arr) == 0:
            return
        if len(arr) < 3:
            off = half_w / 2 if side == 'right' else -half_w / 2
            for v in arr:
                ax.scatter(xpos + off, v, color=color, s=40, zorder=6, lw=0, alpha=0.8)
            return
        yg = _npe.linspace(max(arr.min() - 0.5, 0), arr.max() + 0.5, 200)
        try:
            sig = _npe.std(arr); bw = max(0.5, sig * 0.4)
            kde = _gkde_e(arr, bw_method=bw / (sig if sig > 0 else 1))
            dn  = kde(yg); dn = dn / dn.max() * half_w if dn.max() > 0 else dn
        except Exception:
            return
        if side == 'right':
            ax.fill_betweenx(yg, xpos, xpos + dn, color=color, alpha=0.45, lw=0)
            ax.plot(xpos + dn, yg, color=color, lw=0.8, alpha=0.85)
        else:
            ax.fill_betweenx(yg, xpos - dn, xpos, color=color, alpha=0.45, lw=0)
            ax.plot(xpos - dn, yg, color=color, lw=0.8, alpha=0.85)
        med    = float(_npe.median(arr))
        med_dn = float(_npe.interp(med, yg, dn))
        if side == 'right':
            ax.plot([xpos, xpos + med_dn], [med, med], color=color, lw=2.0, zorder=6)
        else:
            ax.plot([xpos - med_dn, xpos], [med, med], color=color, lw=2.0, zorder=6)

    for _i_e, _cid_e in enumerate(_cids_e):
        _ri_e, _ci_e = _i_e // _NCE, _i_e % _NCE
        ax = fig_e.add_subplot(_gse[_ri_e, _ci_e])
        _rows_e = _vc3[_vc3['cluster25'] == _cid_e]
        _n_e    = len(_rows_e)
        _dv_e   = _rows_e['era5_day_n_persistent_events'].dropna().values
        _nv_e   = _rows_e['era5_night_n_persistent_events'].dropna().values

        _sd_de = float(_npe.std(_dv_e)) if len(_dv_e) >= 2 else float('nan')
        _sd_ne = float(_npe.std(_nv_e)) if len(_nv_e) >= 2 else float('nan')
        _sd_se = (f'  σ_d={_sd_de:.1f}, σ_n={_sd_ne:.1f} days'
                  if not (_npe.isnan(_sd_de) or _npe.isnan(_sd_ne)) else '')

        _draw_half_e(ax, _dv_e, 1.5, 'right', _DAY_E)
        _draw_half_e(ax, _nv_e, 1.5, 'left',  _NGT_E)

        ax.set_title(f'Cluster {_cid_e:02d}  (n={_n_e}){_sd_se}',
                     fontsize=6.5, fontweight='bold', pad=3)
        ax.set_ylabel('ERA5 Persistent Events', fontsize=6)
        ax.set_xticks([1.5]); ax.set_xticklabels(['◄ Night  |  Day ►'], fontsize=6)
        ax.set_xlim(0.8, 2.2)
        ax.tick_params(axis='y', labelsize=5)
        ax.grid(axis='y', alpha=0.25, lw=0.4)
        ax.spines[['top', 'right']].set_visible(False)
        if _i_e == 0:
            ax.legend(handles=[_leg_e_d, _leg_e_n], fontsize=5,
                      loc='upper right', framealpha=0.6)

    for _j_e in range(len(_cids_e), _NRE * _NCE):
        fig_e.add_subplot(_gse[_j_e // _NCE, _j_e % _NCE]).set_visible(False)

    fig_e.savefig(_OUT_8E, bbox_inches='tight', dpi=config.FIGURE_DPI)
    _plte.close(fig_e)
    print(f'  ✓ {_OUT_8E.name}  ({len(_cids_e)} clusters, {_NRE}×{_NCE})')

# ── §8e  MODIS + combined butterfly — removed (MODIS validation-only) ─────────────
print('  §8e MODIS/combined butterfly — skipped (removed)')

<a id="nb04-s9"></a>
## §9 — SEVIRI Download

We download MSG-SEVIRI LST as one national mosaic per event window (day 13–15 UTC, night 01–03 UTC)
and clip each city's region from it.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# -- §9 — SEVIRI download via process_all_cities_seviri()
# Uses load_city_event_periods() which falls back to per-city audit JSONs.
# output_dir = PROJECT_ROOT / 'Results' so the function finds
#   Results/04_temporal_analysis_daily/events/{city}_event_audit.json

# ── Test mode control ─────────────────────────────────────────────────────
# Set SEVIRI_TEST_MODE = True to run a single city end-to-end first.
# Once the pipeline is confirmed correct, set to False for the full run.
SEVIRI_TEST_MODE  = False
SEVIRI_TEST_CITIES = ['Abuja']   # any city that has a selected_event audit
# ──────────────────────────────────────────────────────────────────────────

from src import seviri_processing_utils
importlib.reload(seviri_processing_utils)

_SEVIRI_CREDS_FILE = PROJECT_ROOT / 'secrets' / 'eumetsat_credentials.json'
_SEVIRI_AVAILABLE = _SEVIRI_CREDS_FILE.exists()

if not _SEVIRI_AVAILABLE:
    print('WARN: EUMETSAT credentials not found. SEVIRI download skipped.')
    print(f'  Expected at: {_SEVIRI_CREDS_FILE}')
else:
    with open(_SEVIRI_CREDS_FILE) as _f:
        _eumetsat_creds = json.load(_f)

    _SEVIRI_TEMP = PROJECT_ROOT / 'data' / 'tmp' / 'seviri_temp'
    _SEVIRI_TEMP.mkdir(parents=True, exist_ok=True)

    _seviri_output_dir = str(PROJECT_ROOT / 'Results')
    _seviri_aoi_dir    = str(PROJECT_ROOT / 'Results' / '01_aoi_delineation')

    print('Starting SEVIRI processing...')
    if SEVIRI_TEST_MODE:
        print(f'  TEST MODE: {SEVIRI_TEST_CITIES}')
    print(f'  output_dir : {_seviri_output_dir}')
    print(f'  aoi_dir    : {_seviri_aoi_dir}')
    print()

    _seviri_summary = seviri_processing_utils.process_all_cities_seviri(
        eumetsat_creds = _eumetsat_creds,
        output_dir     = _seviri_output_dir,
        study_year     = STUDY_YEAR,
        temp_dir       = str(_SEVIRI_TEMP),
        aoi_dir        = _seviri_aoi_dir,
        city_list      = CITIES,
        n_workers      = 2,
        test_mode      = SEVIRI_TEST_MODE,
        test_cities    = SEVIRI_TEST_CITIES,
    )

    if _seviri_summary:
        print(f'  SEVIRI complete: {len(_seviri_summary)} cities processed.')
        if SEVIRI_TEST_MODE:
            print()
            print('  TEST PASSED. Set SEVIRI_TEST_MODE = False to run all cities.')
    else:
        print('  SEVIRI processing returned no results.')

<a id="nb04-s10"></a>
## §10 — SEVIRI Three-Zone Analysis + SUHII

For each city, period and selected event window we clip the mosaic to the city region, take the median
LST of the UC, BRA and CR, and compute SUHII = UC − BRA, then aggregate to clusters and the nation.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── §10 — SEVIRI 3-zone extraction (UC / BRA / CR) + SUHII ─────────────────────
# Reads per-city clipped TIFs produced by §9 (process_all_cities_seviri).
# Each city/period has its own event window and its own clipped TIF at:
#   SEVIRI_DIR / {city_slug} / {city_slug}_SEVIRI_{period}_{start}_{end}_median_102022_1km_clipped.tif

ZONE_CSV = SEVIRI_DIR / 'City_SEVIRI_3zone_SUHII.csv'

if SKIP_EXISTING and ZONE_CSV.exists():
    df_seviri = pd.read_csv(ZONE_CSV)
    print(f'Loaded cached {ZONE_CSV.name}  ({len(df_seviri)} rows)')
else:
    try:
        from rasterstats import zonal_stats as _zs
        import geopandas as _gpd_zs
        # All SEVIRI TIFs are ESRI:102022; geometries are WGS84.
        # Reproject to match raster CRS; use spatial mean (SUHII convention).
        def _zonal_mean(geom, tif_path):
            if geom is None or not Path(tif_path).exists():
                return None
            try:
                geom_repr = (
                    _gpd_zs.GeoDataFrame(geometry=[geom], crs='EPSG:4326')
                    .to_crs('ESRI:102022')
                    .geometry.iloc[0]
                )
                stats = _zs([geom_repr.__geo_interface__], str(tif_path), stats=['mean'])
                return stats[0]['mean'] if stats else None
            except Exception:
                return None
    except ImportError:
        _zs = None
        def _zonal_mean(geom, tif_path):
            return None

    _seviri_rows = []
    _pairs = [(c, p) for c in CITIES for p in ('day', 'night')]
    _pbar_z = tqdm(_pairs, desc='SEVIRI 3-zone', unit='city-period')

    for city, period in _pbar_z:
        _pbar_z.set_postfix_str(f'{city[:18]} [{period}]')

        # Each period uses its own independently selected event window
        audit = _audit_day.get(city, {}) if period == 'day' else _audit_night.get(city, {})
        event = audit.get('selected_event')
        if event is None:
            continue

        start     = str(event.get('start_date', ''))
        end       = str(event.get('end_date', ''))
        tier      = event.get('tier', '')
        start_cmp = start.replace('-', '')   # '2024-01-19' -> '20240119'
        end_cmp   = end.replace('-', '')

        # Folder and filename use canonical city name (spaces, not underscores)
        clipped_tif = (SEVIRI_DIR / city /
                       f'{city}_SEVIRI_{period}_{start_cmp}_{end_cmp}_median_102022_1km_clipped.tif')

        uc_lst  = _zonal_mean(_uc_geom.get(city),  clipped_tif)
        bra_lst = _zonal_mean(_bra_geom.get(city), clipped_tif)
        cr_lst  = _zonal_mean(_cr_geom.get(city),  clipped_tif)
        suhii   = (uc_lst - bra_lst) if (uc_lst is not None and bra_lst is not None) else None

        if not clipped_tif.exists():
            note = 'tif_not_downloaded'
        elif _zs is None:
            note = 'rasterstats_not_installed'
        else:
            note = 'ok'

        _seviri_rows.append({
            'city':          city,
            'period':        period,
            'start_date':    start,
            'end_date':      end,
            'uc_lst_mean':  round(uc_lst,  2) if uc_lst  is not None else None,
            'bra_lst_mean': round(bra_lst, 2) if bra_lst is not None else None,
            'cr_lst_mean':  round(cr_lst,  2) if cr_lst  is not None else None,
            'suhii':          round(suhii,   3) if suhii   is not None else None,
            'tier':           tier,
            'note':           note,
        })

    _pbar_z.close()
    df_seviri = pd.DataFrame(_seviri_rows)
    df_seviri.to_csv(ZONE_CSV, index=False)
    print(f'  Saved {ZONE_CSV.name}  ({len(df_seviri)} rows)')

# Summary table
def _fmt(v): return f'{v:.2f}' if v is not None else 'N/A'
_downloaded = df_seviri[df_seviri['note'] == 'ok']
_pending    = df_seviri[df_seviri['note'] == 'tif_not_downloaded']
print(f'  Downloaded & extracted : {len(_downloaded)} city-period rows')
print(f'  Pending download       : {len(_pending)} city-period rows')
print()
print(f"{'City':22s} {'Period':7s} {'UC':>8s} {'BRA':>8s} {'CR':>8s} {'SUHII':>8s} {'Tier':>10s}  Note")
print('-' * 90)
for _, r in df_seviri.iterrows():
    print(
        f"  {r['city']:20s} {r['period']:7s} {_fmt(r['uc_lst_mean']):>8s} "
        f"{_fmt(r['bra_lst_mean']):>8s} {_fmt(r['cr_lst_mean']):>8s} "
        f"{_fmt(r['suhii']):>8s} {str(r.get('tier','—')):>10s}  {r.get('note','')}"
    )


<a id="nb04-s10b"></a>
### §10b — Cluster-Level SUHII Aggregation

We average the city SUHII of each cluster, day and night.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── §10b — Cluster-level SUHII aggregation ────────────────────────────────────
# Only aggregate cities that have been downloaded and extracted (note == 'ok').
# SUHII = mean(UC) - mean(BRA); statistics across cities within each cluster.
_df_sev_ok = df_seviri[df_seviri['note'] == 'ok'].copy()
_df_sev_ok['cluster_num'] = _df_sev_ok['city'].map(_city_to_cluster)

if _df_sev_ok.empty:
    print('No SEVIRI data extracted yet — download more cities first.')
    df_cl_suhii = pd.DataFrame()
else:
    df_cl_suhii = (
        _df_sev_ok.dropna(subset=['cluster_num'])
        .groupby(['cluster_num', 'period'])
        .agg(
            n_cities     = ('city',        'nunique'),
            uc_mean      = ('uc_lst_mean',  'mean'),
            bra_mean     = ('bra_lst_mean', 'mean'),
            cr_mean      = ('cr_lst_mean',  'mean'),
            suhii_mean   = ('suhii',        'mean'),
            suhii_std    = ('suhii',        'std'),
            suhii_min    = ('suhii',        'min'),
            suhii_max    = ('suhii',        'max'),
        )
        .reset_index()
        .round(3)
    )
    print(f'Cluster-level SUHII ({len(_df_sev_ok)} city-periods with data, '
          f'{_df_sev_ok["city"].nunique()} cities):')
    print()
    for period in ('day', 'night'):
        sub = df_cl_suhii[df_cl_suhii['period'] == period]
        if sub.empty:
            continue
        print(f'  {period.upper()}:')
        print(f"  {'Cluster':>8s}  {'N':>3s}  {'UC':>7s}  {'BRA':>7s}  "
              f"{'SUHII_mean':>10s}  {'SUHII_std':>9s}  {'SUHII_min':>9s}  {'SUHII_max':>9s}")
        print('  ' + '-' * 72)
        for _, r in sub.iterrows():
            def _f(v): return f'{v:.2f}' if pd.notna(v) else 'N/A'
            print(f"  {int(r['cluster_num']):>8d}  {int(r['n_cities']):>3d}  "
                  f"{_f(r['uc_mean']):>7s}  {_f(r['bra_mean']):>7s}  "
                  f"{_f(r['suhii_mean']):>10s}  {_f(r['suhii_std']):>9s}  "
                  f"{_f(r['suhii_min']):>9s}  {_f(r['suhii_max']):>9s}")
        print()
    df_cl_suhii.to_csv(SEVIRI_DIR / 'Cluster_SEVIRI_SUHII.csv', index=False)
    print('  ✓ Cluster_SEVIRI_SUHII.csv')


---
<a id="nb04-s10c"></a>
### §10c — Master Composite Summary Table

We summarise all NB04 results in one table: city rows, single-city clusters merged (◆), cluster
composites (▶) after their cities, and the national row (■) last. Its columns hold the identification
and event tier; the MODIS LST, hot days and persistent events, day and night; the selected events; the
ERA5–MODIS correlation; and the SEVIRI SUHII mean and maximum (N/A if §10 has not run). We skip it when
both files exist.

**Outputs:** `Stats/NB04_Master_Composite_Summary.csv` and `Stats/NB04_Master_Composite_Summary.html`

> ↑ [Back to Table of Contents](#nb04-toc)

In [ ]:
# ── §10c — Master composite summary table ────────────────────────────────────
import numpy as _npm

_STATS_DIR = RESULTS_04 / 'Stats'
_STATS_DIR.mkdir(parents=True, exist_ok=True)
_OUT_CSV  = _STATS_DIR / 'NB04_Master_Composite_Summary.csv'
_OUT_HTML = _STATS_DIR / 'NB04_Master_Composite_Summary.html'

if _OUT_CSV.exists() and _OUT_HTML.exists():
    print(f'§10c cache hit — {_OUT_CSV.name}')
else:
    # ── Load SEVIRI SUHII if available ────────────────────────────────────────
    _suhii_city_csv = RESULTS_04 / 'seviri' / 'city_suhii_summary.csv'
    _SEV_COLS = ['suhii_day_mean', 'suhii_day_max', 'suhii_day_min',
                 'suhii_day_trend', 'suhii_night_mean', 'suhii_night_max',
                 'suhii_night_min', 'suhii_night_trend']
    if _suhii_city_csv.exists():
        _df_suhii    = pd.read_csv(_suhii_city_csv)
        _suhii_avail = True
        print(f'  SEVIRI SUHII: {len(_df_suhii)} city rows loaded')
    else:
        _df_suhii    = pd.DataFrame()
        _suhii_avail = False
        print('  SEVIRI SUHII: not yet available (§10 pending)')

    # ── Core columns from df_city_heat ────────────────────────────────────────
    _CORE = [
        'city', 'cluster25', 'validation_status',
        # ERA5 primary LST stats
        'era5_day_lst_mean_C',  'era5_night_lst_mean_C',
        'era5_day_lst_max_C',   'era5_night_lst_max_C',
        'era5_day_p90_threshold_C', 'era5_night_p90_threshold_C',
        'era5_day_n_persistent_events', 'era5_night_n_persistent_events',
        'era5_day_longest_event_days',  'era5_night_longest_event_days',
        # MODIS comparison
        'modis_day_lst_mean_C', 'modis_night_lst_mean_C',
        'modis_day_lst_max_C',  'modis_night_lst_max_C',
        'modis_day_n_hot_days', 'modis_night_n_hot_days',
        'day_n_era5_persist_evts', 'night_n_era5_persist_evts',
        'day_era5_longest_days',   'night_era5_longest_days',
        'day_era5_shortest_days',  'night_era5_shortest_days',
        'day_era5_mean_event_days','night_era5_mean_event_days',
        'day_era5_std_event_days', 'night_era5_std_event_days',
        'day_n_modis_persist_evts','night_n_modis_persist_evts',
        'day_modis_longest_days',  'night_modis_longest_days',
        'day_modis_shortest_days', 'night_modis_shortest_days',
        'day_modis_mean_event_days','night_modis_mean_event_days',
        'day_modis_std_event_days', 'night_modis_std_event_days',
        'day_sel_start', 'day_sel_end', 'day_sel_duration_days', 'day_sel_tier',
        'night_sel_start','night_sel_end','night_sel_duration_days','night_sel_tier',
        'day_corr_r', 'day_corr_r2', 'day_corr_bias', 'day_corr_rmse',
        'night_corr_r','night_corr_r2','night_corr_bias','night_corr_rmse',
    ]
    _avail = [c for c in _CORE if c in df_city_heat.columns]
    _base  = df_city_heat[_avail].copy()
    _base['cluster25'] = _base['cluster25'].astype('Int64')

    if _suhii_avail and 'city' in _df_suhii.columns:
        _sc = [c for c in _SEV_COLS if c in _df_suhii.columns]
        _base = _base.merge(_df_suhii[['city'] + _sc], on='city', how='left')
    else:
        for col in _SEV_COLS:
            _base[col] = 'N/A'

    _num_cols = [c for c in _base.select_dtypes('number').columns
                 if c not in ('cluster25',)]

    # ── Build rows ────────────────────────────────────────────────────────────
    _table_rows = []

    for cid in sorted(_base['cluster25'].dropna().unique()):
        _sub = _base[_base['cluster25'] == cid].sort_values('city')
        _n_c = len(_sub)

        if _n_c == 1:
            row = _sub.iloc[0].to_dict()
            row['row_type']    = '◆ City/Cluster'
            row['display_name']= f"◆ {row['city']}  (C{int(cid):02d})"
            _table_rows.append(row)
        else:
            for _, cr in _sub.iterrows():
                row = cr.to_dict()
                row['row_type']    = 'City'
                row['display_name']= f"  {row['city']}"
                _table_rows.append(row)

            # Cluster composite row from df_cl_heat
            if 'df_cl_heat' in dir() or 'df_cl_heat' in globals():
                _cmr = df_cl_heat[df_cl_heat['cluster25'] == cid]
                if not _cmr.empty:
                    _cr  = _cmr.iloc[0]
                    crow = {
                        'display_name': f'▶ Cluster {int(cid):02d}  (n={_n_c})',
                        'row_type':     '▶ Cluster',
                        'city':         f'Cluster {int(cid):02d}',
                        'cluster25':    int(cid),
                        'validation_status': _cr.get('modal_validation_tier'),
                    }
                    for col in _num_cols:
                        mean_col = f'{col}_mean'
                        if mean_col in df_cl_heat.columns:
                            crow[col] = df_cl_heat[
                                df_cl_heat['cluster25'] == cid
                            ][mean_col].values[0]
                        else:
                            crow[col] = None
                    for sev_c in _SEV_COLS:
                        crow[sev_c] = None
                    _table_rows.append(crow)

    # National row
    _nat = {'display_name': '■ NATIONAL', 'row_type': '■ National',
            'city': 'NATIONAL', 'cluster25': None, 'validation_status': None}
    for col in _num_cols:
        vals = _base[col].dropna()
        try:
            _nat[col] = round(float(vals.mean()), 3) if len(vals) else None
        except Exception:
            _nat[col] = None
    _table_rows.append(_nat)

    # ── Save CSV + HTML ───────────────────────────────────────────────────────
    df_master = pd.DataFrame(_table_rows)
    _front = ['display_name', 'row_type', 'cluster25', 'validation_status']
    df_master = df_master[_front + [c for c in df_master.columns if c not in _front]]
    df_master.to_csv(_OUT_CSV, index=False)
    print(f'  ✓ {_OUT_CSV.name}  ({len(df_master)} rows, {len(df_master.columns)} cols)')

    _sty = {'◆ City/Cluster': 'background:#fff3cd;font-weight:bold',
            '▶ Cluster':      'background:#d4edda;font-weight:bold',
            '■ National':     'background:#cce5ff;font-weight:bold',
            'City': ''}
    _html = ['<table border="1" cellspacing="0" cellpadding="3" '
             'style="font-size:9px;border-collapse:collapse">',
             '<tr>' + ''.join(
                 f'<th style="background:#333;color:#fff;padding:4px">{c}</th>'
                 for c in df_master.columns) + '</tr>']
    for _, r in df_master.iterrows():
        _s = _sty.get(str(r.get('row_type', '')), '')
        _html.append(
            f'<tr style="{_s}">' +
            ''.join(f'<td style="padding:2px">{r[c] if pd.notna(r[c]) else "—"}</td>'
                    for c in df_master.columns) + '</tr>')
    _html.append('</table>')
    _OUT_HTML.write_text(''.join(_html), encoding='utf-8')
    print(f'  ✓ {_OUT_HTML.name}')
    _row_s = df_master['row_type'].value_counts()
    print(f'  City: {_row_s.get("City",0)}  ◆merged: {_row_s.get("◆ City/Cluster",0)}  '
          f'▶cluster: {_row_s.get("▶ Cluster",0)}  ■national: 1')

<a id="nb04-s10d"></a>
### §10d — National Composite SUHII

We average the city SUHII over all study cities, day and night, and chart each city's SUHII by cluster.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── §10d — National composite SUHII ──────────────────────────────────────────
# Statistics computed across all downloaded cities.
_df_sev_ok = df_seviri[df_seviri['note'] == 'ok'].copy()

if _df_sev_ok.empty:
    print('No SEVIRI data extracted yet — download more cities first.')
    df_nat_suhii = pd.DataFrame()
else:
    df_nat_suhii = (
        _df_sev_ok
        .groupby('period')
        .agg(
            n_cities     = ('city',        'nunique'),
            uc_mean      = ('uc_lst_mean',  'mean'),
            bra_mean     = ('bra_lst_mean', 'mean'),
            cr_mean      = ('cr_lst_mean',  'mean'),
            suhii_mean   = ('suhii',        'mean'),
            suhii_std    = ('suhii',        'std'),
            suhii_min    = ('suhii',        'min'),
            suhii_max    = ('suhii',        'max'),
        )
        .reset_index()
        .round(3)
    )
    print(f'National composite SUHII ({_df_sev_ok["city"].nunique()} cities):')
    print()
    def _f(v): return f'{v:.2f}' if pd.notna(v) else 'N/A'
    print(f"  {'Period':>7s}  {'N':>3s}  {'UC_mean':>8s}  {'BRA_mean':>8s}  "
          f"{'CR_mean':>8s}  {'SUHII_mean':>10s}  {'Std':>6s}  {'Min':>6s}  {'Max':>6s}")
    print('  ' + '-' * 76)
    for _, r in df_nat_suhii.iterrows():
        print(f"  {r['period']:>7s}  {int(r['n_cities']):>3d}  "
              f"{_f(r['uc_mean']):>8s}  {_f(r['bra_mean']):>8s}  {_f(r['cr_mean']):>8s}  "
              f"{_f(r['suhii_mean']):>10s}  {_f(r['suhii_std']):>6s}  "
              f"{_f(r['suhii_min']):>6s}  {_f(r['suhii_max']):>6s}")
    print()
    df_nat_suhii.to_csv(SEVIRI_DIR / 'National_SEVIRI_SUHII.csv', index=False)
    print('  ✓ National_SEVIRI_SUHII.csv')


In [ ]:
# ── §10d — City-level SUHII chart (cluster-colour bars) ──────────────
importlib.reload(visualization)
from src.visualization import create_suhii_city_chart

# Bar fill  = Cluster25 colour
# Cities sorted by daytime SUHII descending (highest at top).

fig_suhii = create_suhii_city_chart(
    df_seviri      = df_seviri,
    cluster_lookup = _city_to_cluster,
    output_dir     = str(CHARTS_DIR),
    dpi            = config.FIGURE_DPI,
)
if fig_suhii:
    plt.show()
    print(f'  ✓ SEVIRI_SUHII_cities.svg / .pdf  → {CHARTS_DIR}')
else:
    print('  No valid SEVIRI SUHII data yet — download cities first.')

<a id="nb04-s11"></a>
## §11 — SEVIRI vs ERA5 Correlation (Event Window)

We compare the SEVIRI and ERA5-Land urban-core LST over each selected event window only, with r², the
1:1 line and a regression, per city and cluster. We leave MODIS out here: cloud leaves too few MODIS
days within a window.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── §11 SEVIRI vs ERA5 event-window correlation ───────────────────────────────
print('SEVIRI vs ERA5-Land event-window correlation')
print('=' * 70)

_sev_era5_rows = []

for city in CITIES:
    for period in ('day', 'night'):
        audit = _audit_day.get(city, {}) if period == 'day' else _audit_night.get(city, {})
        sel   = audit.get('selected_event')
        if sel is None:
            continue

        start = pd.to_datetime(sel['start_date'])
        end   = pd.to_datetime(sel['end_date'])

        # ERA5 mean over event window
        era5_csv = RAW_DIR / f'{city}_ERA5_UC_daily_{STUDY_YEAR}.csv'
        if not era5_csv.exists():
            continue
        edf_p = load_daily_csv(era5_csv, period)
        if edf_p is None:
            continue
        e_win     = edf_p[(edf_p['date'] >= start) & (edf_p['date'] <= end)]['lst_median']
        era5_mean = float(e_win.mean()) if not e_win.empty else None

        # SEVIRI UC median from df_seviri (computed in §10)
        sev_row = df_seviri[
            (df_seviri['city'] == city) & (df_seviri['period'] == period)
        ]
        sev_uc = float(sev_row['uc_lst_mean'].iloc[0]) if not sev_row.empty else None

        # Guard: skip if either value is missing or NaN (city not yet downloaded)
        if era5_mean is None or sev_uc is None or pd.isna(era5_mean) or pd.isna(sev_uc):
            continue

        tier = sel.get('tier', 'FAIR')
        _sev_era5_rows.append({
            'city':        city,
            'period':      period,
            'era5_mean':   round(era5_mean, 2),
            'seviri_mean': round(sev_uc, 2),
            'tier':        tier,
        })
        print(f'  {city:20s} [{period:5s}]  ERA5={era5_mean:.1f}°C  SEVIRI={sev_uc:.1f}°C  '
              f'diff={sev_uc - era5_mean:+.1f}°C  [{tier}]')

df_sev_era5 = pd.DataFrame(_sev_era5_rows)

if df_sev_era5.empty:
    print('\nNo matched SEVIRI+ERA5 pairs yet — download more cities first.')
else:
    importlib.reload(visualization)
    from src.visualization import create_seviri_era5_scatter
    _day_rows   = [r for r in _sev_era5_rows if r['period'] == 'day']
    _night_rows = [r for r in _sev_era5_rows if r['period'] == 'night']
    fig_sc = create_seviri_era5_scatter(
        day_rows    = _day_rows,
        night_rows  = _night_rows,
        zone        = 'urban_core',
        output_dir  = str(CORR_DIR),
        dpi         = config.FIGURE_DPI,
    )
    if fig_sc:
        plt.show()
        print('  ✓ SEVIRI_vs_ERA5_scatter_combined.svg / .pdf')

    # [ALTERNATIVE] Separate day / night scatter figures:
    # for period in ('day', 'night'):
    #     sub = [r for r in _sev_era5_rows if r['period'] == period]
    #     create_seviri_era5_scatter(day_rows=sub if period=='day' else [],
    #                                night_rows=sub if period=='night' else [],
    #                                output_dir=str(CORR_DIR),
    #                                output_stem=f'SEVIRI_vs_ERA5_scatter_{period}')

    df_sev_era5.to_csv(CORR_DIR / 'SEVIRI_vs_ERA5_event_window_correlation.csv', index=False)
    print('  ✓ SEVIRI_vs_ERA5_event_window_correlation.csv')

    # Cluster-level SEVIRI vs ERA5
    df_sev_era5['cluster_num'] = df_sev_era5['city'].map(_city_to_cluster)
    df_cl_sev_corr = (
        df_sev_era5.dropna(subset=['cluster_num'])
        .groupby(['cluster_num', 'period'])
        .agg(
            n_cities      = ('city',        'count'),
            era5_range_lo = ('era5_mean',   'min'),
            era5_range_hi = ('era5_mean',   'max'),
            sev_range_lo  = ('seviri_mean', 'min'),
            sev_range_hi  = ('seviri_mean', 'max'),
            mean_diff     = ('seviri_mean', lambda x: (x - df_sev_era5.loc[x.index, 'era5_mean']).mean()),
        )
        .reset_index()
        .round(2)
    )
    df_cl_sev_corr.to_csv(CORR_DIR / 'Cluster_SEVIRI_vs_ERA5_event_window.csv', index=False)
    print('  ✓ Cluster_SEVIRI_vs_ERA5_event_window.csv')
    print()
    print(df_cl_sev_corr.to_string(index=False))


<a id="nb04-s11a"></a>
### §11a — Target Diagram: SEVIRI UC vs ERA5-Land (Event Window)

We draw the Target Diagram of §6a for SEVIRI against ERA5-Land: the SEVIRI urban-core mean (one value
per city and window) against the daily ERA5-Land values of the same window.

| Metric | Formula |
|--------|---------|
| Bias   | SEVIRI_UC − mean(ERA5_event_daily) |
| RMSE   | √ mean[(SEVIRI_UC − ERA5_daily_i)²] across event days |
| uRMSE  | √(RMSE² − Bias²), sign = +1 (SEVIRI is a single pooled value, no anti-correlation possible) |
A point near the origin means close agreement; above the x-axis SEVIRI is warmer than ERA5-Land, below
cooler. Only cities with clipped SEVIRI scenes appear.

---
*[↑ Table of Contents](#nb04-toc)*

In [ ]:
# ── §11a — Target Diagram: SEVIRI UC vs ERA5-Land (event window) ───────────────
importlib.reload(visualization)
from src.visualization import plot_target_diagram

_sev_target_rows = []

for city in CITIES:
    _day_bias = _night_bias = float('nan')
    _day_rmse = _night_rmse = float('nan')
    _day_mae  = _night_mae  = float('nan')
    _day_n_ev = _night_n_ev = 0

    for period in ('day', 'night'):
        audit = _audit_day.get(city, {}) if period == 'day' else _audit_night.get(city, {})
        sel   = audit.get('selected_event')
        if sel is None:
            continue

        # SEVIRI UC mean for this city × period (from §10)
        _sev_row = df_seviri[
            (df_seviri['city'] == city) & (df_seviri['period'] == period)
        ]
        if _sev_row.empty or pd.isna(_sev_row['uc_lst_mean'].iloc[0]):
            continue
        _sev_uc = float(_sev_row['uc_lst_mean'].iloc[0])

        # ERA5 daily values over the same event window
        _era5_csv = RAW_DIR / f'{city}_ERA5_UC_daily_{STUDY_YEAR}.csv'
        if not _era5_csv.exists():
            continue
        _edf = load_daily_csv(_era5_csv, period)
        if _edf is None:
            continue
        _start = pd.to_datetime(sel['start_date'])
        _end   = pd.to_datetime(sel['end_date'])
        _e_win = _edf[(_edf['date'] >= _start) & (_edf['date'] <= _end)]['lst_median']
        if _e_win.empty:
            continue

        _era5_vals = _e_win.to_numpy(dtype=float)
        _bias = _sev_uc - float(_era5_vals.mean())
        _rmse = float(np.sqrt(np.mean((_sev_uc - _era5_vals) ** 2)))
        _mae  = float(np.mean(np.abs(_sev_uc - _era5_vals)))
        _n_ev = int(len(_era5_vals))

        if period == 'day':
            _day_bias, _day_rmse = _bias, _rmse
            _day_mae, _day_n_ev  = _mae, _n_ev
        else:
            _night_bias, _night_rmse = _bias, _rmse
            _night_mae, _night_n_ev  = _mae, _n_ev

    if pd.isna(_day_bias) and pd.isna(_night_bias):
        continue

    _sev_target_rows.append({
        'city':              city,
        'day_bias':          _day_bias,
        'day_rmse':          _day_rmse,
        # r = 1.0 → sign(r) = +1; uRMSE = sqrt(RMSE² - Bias²) on positive x-axis
        'day_mae':           _day_mae,
        'day_n_obs':         _day_n_ev,
        'day_correlation':   1.0,  # SEVIRI scalar vs ERA5 array; positive x-axis by convention
        'night_bias':        _night_bias,
        'night_rmse':        _night_rmse,
        'night_mae':         _night_mae,
        'night_n_obs':       _night_n_ev,
        'night_correlation': 1.0,
    })

if not _sev_target_rows:
    print('No SEVIRI+ERA5 pairs available yet — download more cities first.')
else:
    fig_td_sev, td_sev_summary = plot_target_diagram(
        comparison_results = _sev_target_rows,
        dataset1_name      = 'ERA5-Land',
        dataset2_name      = 'SEVIRI',
        output_dir         = str(CORR_DIR),
        cluster_lookup     = _city_to_cluster,
        show_labels        = True,
    )
    plt.show()

    print()

    # ── Export SEVIRI vs ERA5 metrics to CSV ─────────────────────────────
    _sev_csv_out = CORR_DIR / 'SEVIRI_vs_ERA5_target_metrics.csv'
    _sev_df_out  = pd.DataFrame(_sev_target_rows)[
        ['city', 'day_bias', 'day_rmse', 'day_mae', 'day_n_obs',
                 'night_bias','night_rmse','night_mae','night_n_obs']
    ]
    _sev_df_out.to_csv(_sev_csv_out, index=False)
    print(f'  ✓ SEVIRI vs ERA5 metrics exported → {_sev_csv_out.name}')
    print(f"SEVIRI vs ERA5 — TARGET DIAGRAM SUMMARY  ({td_sev_summary['n_cities']} cities)")
    print("=" * 76)

    for _period in ('day', 'night'):
        _s  = td_sev_summary.get(_period, {})
        if not _s:
            continue
        # Gather per-city values for this period
        _biases = [r[f'{_period}_bias'] for r in _sev_target_rows
                   if not pd.isna(r[f'{_period}_bias'])]
        _rmses  = [r[f'{_period}_rmse'] for r in _sev_target_rows
                   if not pd.isna(r[f'{_period}_rmse'])]

        print(f"  {'─'*72}")
        print(f"  {_period.upper()} PERIOD  ({len(_biases)} cities with data)")
        print(f"  {'─'*72}")
        print(f"  {'Metric':<18s}  {'Mean':>7s}  {'Std':>7s}  {'Min':>7s}  {'Max':>7s}")
        print(f"  {'─'*54}")

        import numpy as _npsum
        def _row(label, vals):
            v = _npsum.array(vals)
            print(f"  {label:<18s}  {_npsum.nanmean(v):>+7.3f}  "
                  f"{_npsum.nanstd(v):>7.3f}  {_npsum.nanmin(v):>+7.3f}  {_npsum.nanmax(v):>+7.3f}")

        _row('Bias  (°C)',  _biases)
        _row('RMSE  (°C)',  _rmses)

        print(f"  Best  agreement : {_s['best_city']}  (RMSE = {_s['best_rmse']:.3f} °C)")
        print(f"  Worst agreement : {_s['worst_city']}  (RMSE = {_s['worst_rmse']:.3f} °C)")

        # Per-city table
        print(f"  {'City':<22s}  {'Bias (°C)':>9s}  {'RMSE (°C)':>9s}  {'Agreement'}")
        print(f"  {'─'*60}")
        _period_rows = sorted(
            [(r['city'], r[f'{_period}_bias'], r[f'{_period}_rmse'])
             for r in _sev_target_rows
             if not pd.isna(r[f'{_period}_rmse'])],
            key=lambda x: abs(x[1])  # sort by |bias|
        )
        for _city, _b, _r in _period_rows:
            _agree = 'Excellent' if _r < 1 else ('Good' if _r < 2 else 'Fair')
            print(f"  {_city:<22s}  {_b:>+9.3f}  {_r:>9.3f}  {_agree}")
    print()

    # ── Export SEVIRI vs ERA5 metrics to CSV ─────────────────────────────
    _sev_csv_out = CORR_DIR / 'SEVIRI_vs_ERA5_target_metrics.csv'
    _sev_df_out  = pd.DataFrame(_sev_target_rows)[
        ['city', 'day_bias', 'day_rmse', 'day_mae', 'day_n_obs',
                 'night_bias','night_rmse','night_mae','night_n_obs']
    ]
    _sev_df_out.to_csv(_sev_csv_out, index=False)
    print(f'  ✓ SEVIRI vs ERA5 metrics exported → {_sev_csv_out.name}')


<a id="nb04-s12"></a>
## §12 — Analysis Summary

### Outputs

```
Results/04_temporal_analysis_daily/
│
├── city_heat_event_summary.csv
├── cluster_heat_event_summary.csv
├── national_heat_event_summary.csv
├── national_daily_LST_summary.csv
├── publication/
│   ├── cluster_daily_composite_2x8.svg
│   ├── cluster_daily_annotation_table.csv
│   ├── heat_event_bars_composite.svg
│   └── heat_event_bars_2x8.svg
├── raw_extractions/
│   ├── {City}_MODIS_UC_daily_2024.csv    ← MODIS UC daily LST (one per city)
│   └── {City}_ERA5_UC_daily_2024.csv     ← ERA5-Land UC daily LST (one per city)
│
├── events/
│   ├── MODIS_hot_event_detection_summary.csv
│   ├── ERA5_hot_event_detection_summary.csv
│   ├── Cluster_event_selection_summary.csv
│   └── {City}_event_audit.json           ← per-city EXCELLENT/GOOD/FAIR audit
│
├── correlation/
│   ├── ERA5_vs_MODIS_full_year_correlation.csv
│   ├── Target_Diagram_MODIS_vs_ERA5-Land.svg
│   ├── SEVIRI_vs_ERA5_scatter_day.svg
│   ├── SEVIRI_vs_ERA5_scatter_night.svg
│   └── SEVIRI_vs_ERA5_event_window_correlation.csv
│       Cluster_SEVIRI_vs_ERA5_event_window.csv
│
├── charts/
│   ├── city_daily_timeseries/
│   │   └── {City}_daily_LST_2024.svg     ← two-panel MODIS/ERA5 per city
│   ├── cluster_daily_composites/
│   │   └── Cluster{NN}_daily_LST_2024.svg
│   ├── Cluster_event_frequency_day.svg
│   └── Cluster_event_frequency_night.svg
│
├── cluster_composites/
│   ├── Cluster{NN}_MODIS_composite_2024.csv
│   └── Cluster{NN}_ERA5_composite_2024.csv
│
└── seviri/
    ├── mosaics/                          ← national SEVIRI mosaics (per window)
    ├── City_SEVIRI_3zone_SUHII.csv
    ├── Cluster_SEVIRI_SUHII.csv
    └── National_SEVIRI_SUHII.csv
```

---
*[↑ Table of Contents](#nb04-toc)*